# [실습] 시간당 수요 차수 결정 — AIC 비교표 6조합

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 0. 오늘 쓸 데이터와 첫날에 정해진 설정 3개

<div style="background:#FFFFFF;color:#000000;padding:40px;border-radius:12px;border:1px solid #E6E6E6;border-left:4px solid #051C2C">
<div style="font-size:0.9em;color:#656565">시계열 예측 · 둘째 날</div>
<h1 style="color:#000000;margin:10px 0">168시간 배차표 — 모형의 차수 2개 결정하기</h1>
<div style="font-size:1.05em;color:#656565">첫날에 보정한 계열로 다음 주 168시간을 채울 모형의 차수를 정하고, 그 근거를 AIC 표 한 장으로 남기는 순서입니다</div>
</div>

## 오늘 할 일

아침에 배차팀장이 다음과 같이 요청했습니다. **"다음 주 트럭 배차표를 금요일까지 결정해야 합니다. 요일별이 아니라 시간대별로, 다음 7일 168시간의 수요를 숫자로 주세요."**

그래서 오늘의 질문은 하나입니다. **그 168시간을 무엇으로 채울 것인가.** 먼저 하루 합계 계열로 대신할 수 있는지 확인하고, 시간 단위 모형의 차수를 결정합니다. 그 차수로 168시간을 채워 기준선과 오차를 비교하는 것은 내일 합니다.

오늘 사용할 설정 3개는 첫날의 진단에서 이미 결정되었습니다. 계절 주기 m = 24, 계절차분 D = 1, 레벨 차분 d = 0 입니다. 오늘 결정할 것은 비계절 차수 p 와 q 2개입니다. 레벨 차분 d 는 후보 전체에 같은 값으로 통일하고, 계절차분은 첫날에 정한 대로 D = 1 입니다.

미션은 2개이고, 그중 차수를 정하는 미션 2 가 오늘 반드시 남겨야 할 결과입니다. 미션 1 은 시간이 남을 때 합니다. 맨 끝의 「오늘의 정리」는 미션 2 까지만 마친 상태에서도 읽을 수 있습니다. 코드 작성은 대부분 AI 튜터에게 맡기고, 우리는 **시킬 일을 결정하고 받은 결과를 확인하는 역할**을 담당합니다.

<img src="https://otexts.com/fpp3/fpp_files/figure-html/auscafecombPI-1.png" width="640" height="320"/>
<p align="center"><em>▲ 호주 월별 외식 지출액에 대해 만든 예측 — 검은 선이 실측, 파란 선이 예측이고 짙은 영역이 80% · 옅은 영역이 95% 예측구간이다. 내일 만들 그림에는 이 중 95% 예측구간 하나만 있다 (출처: Hyndman &amp; Athanasopoulos, Forecasting: Principles and Practice 3rd ed., otexts.com/fpp3)</em></p>

## 오늘의 데이터

**Seoul Bike Sharing Demand** 는 UCI 머신러닝 저장소가 공개한 서울 공유자전거의 시간당 대여 기록입니다. 한 행이 한 시간이고, 2017년 12월 1일 0시부터 2018년 11월 30일 23시까지 8,760행 × 14열입니다. 빠진 시각은 없습니다.

기록 안에는 시스템이 정지해 대여가 한 대도 기록되지 않은 시간이 295개 있습니다. 운영 여부 열 `functioning` 이 `"No"` 인 시각이고, 이하 **휴무 295시간**이라 부릅니다. 그 0 은 수요가 없었다는 뜻이 아니라 관측이 없다는 뜻입니다.

그래서 「준비 셀」은 원본 시간 계열 `count` 를 그대로 두고, 휴무 295시간을 1주일 전 같은 시각 값으로 채운 **보정 수요** `demand` 를 새 열로 따로 만듭니다. 첫날에 정한 대로 **탐색과 검정은 원본 시간 계열, 모형 학습과 예측과 평가는 보정 수요**입니다. 첫날에 만든 파일을 이어받지는 않고, 미션마다 「준비 셀」이 이 zip 을 직접 열어 사용할 변수를 처음부터 만듭니다.

한 시간에 한 값씩 있는 계열은 산문에서 **시간당 수요**로 부릅니다. 코드와 채점 라벨에서는 원본 시간 계열 `count` 입니다.

<img src="https://commons.wikimedia.org/wiki/Special:FilePath/Bicycle-sharing%20station%20at%20Hangang%20bikeway%202.jpg?width=1600" width="640" height="467"/>
<p align="center"><em>▲ 거치대에 정렬된 공유자전거 — 오늘 만들 168시간 표가 결정하는 것은 「몇 대를 몇 시에 채워 두는가」다 (출처: Wikimedia Commons, Bicycle-sharing station at Hangang bikeway)</em></p>

## 이 노트북을 쓰는 법

**미션마다 코드 셀이 2개입니다.** 앞의 것은 사용할 변수를 만드는 셀이라 그대로 실행하고, 뒤의 것은 `PROMPT` 를 내 말로 적어 AI 튜터에게 시키는 셀입니다. 앞을 **「준비 셀」**, 뒤를 **「PROMPT 셀」** 이라 부릅니다. AI 가 작성한 코드는 `# --- 여기부터 AI 코드 ---` 표시선 아래에 들어옵니다.

**출력 이름 규칙** — 「만들 것」에 굵게 적힌 이름을 글자 그대로 사용해 「이름 = 값 단위」 한 행으로 출력합니다. 변수 이름은 자유이며, 채점 기준은 이 출력 행과 그림입니다.

| 무엇 | 어떻게 |
|---|---|
| 실행 환경 | CPU 로 충분합니다. ▶ 실행은 Google Colab 연결 안내를 따르고, AI 튜터는 상단 「Claude 도우미」를 설치한 뒤에 사용합니다. **채점은 Colab 이 연결되면 자동으로 준비됩니다. 코드 패널 위쪽에 「채점 준비됨」이 표시된 뒤에 미션을 시작하고, 미션 셀을 실행했는데 채점 결과가 출력되지 않으면 그 「채점 준비됨」을 누르세요.** 노트북을 내려받아 Colab 이나 내 컴퓨터에서 열었다면 맨 위의 접힌 `[채점]` 셀을 한 번 ▶ 실행한 뒤 시작합니다(코드는 접혀 있어도 실행됩니다). |
| 먼저 실행할 셀 | 맨 위 설치·폰트 셀(한 번이면 됩니다). 설치 뒤 numpy 관련 오류가 발생하면 런타임을 다시 시작하고 이 셀부터 다시 실행합니다 |
| 채점 | 실행하는 순간 실행 결과에 채점 결과가 출력됩니다. 채점 기준을 모두 충족하면 완성본 풀이가 열리고, 충족하지 못하면 어긋난 항목과 AI 튜터에 보낼 질문이 나옵니다 |
| 막혔을 때 | 같은 말을 두 번 그대로 다시 쓰지 말고 `PROMPT` 에 빠진 조건을 하나 추가합니다. 그래도 안 되면 앞 대화를 이어서 쓰지 말고 요구를 한 덩어리로 다시 적어 새 메시지로 보냅니다 |

In [ ]:
# 실습 준비 - 패키지 설치
!pip install -q "numpy>=1.26" "pandas>=2.0" "matplotlib>=3.7" "statsmodels==0.14.6" "pmdarima==2.1.1"

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox
import warnings; warnings.filterwarnings("ignore")

# 한글 폰트 — Colab 이면 나눔고딕을 설치한 뒤 그 ttf 를 폰트 목록에 **직접** 등록한다.
#  설치는 디스크에 파일만 만든다. 이미 떠 있는 폰트 목록은 그 사실을 모르므로 addfont 로
#  넣어 줘야 이름이 보인다(넣지 않으면 DejaVu Sans 로 밀려 한글이 전부 □ 로 나온다).
import glob, shutil, subprocess
if shutil.which("apt-get"):
    subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"], check=False)
for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):   # 설치 여부와 무관하게 한 번
    try: fm.fontManager.addfont(_p)
    except Exception: pass
try: fm._get_font.cache_clear()
except Exception: pass
installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((n for n in ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"] if n in installed), "DejaVu Sans")
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = FONT in {"Pretendard", "DejaVu Sans"}   # 나머지 폰트에는 U+2212 가 없다
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}   # 그림마다 폭이 깎이지 않게
plt.rcParams["figure.dpi"] = plt.rcParams["savefig.dpi"] = 150       # 폭 FIG_W x 150dpi = 1500px 고정
plt.rcParams["figure.autolayout"] = True
plt.rcParams.update({"font.size": 11, "axes.titlesize": 15, "axes.labelsize": 12,
                     "legend.fontsize": 11, "figure.titlesize": 17, "font.weight": "regular",
                     "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
                     "axes.edgecolor": "#757575", "xtick.color": "#757575", "ytick.color": "#757575",
                     "xtick.labelcolor": "black", "ytick.labelcolor": "black",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})

# 오늘 그림 전체가 함께 쓰는 색 — 기준선=남색, 관측=파랑, 강조=하늘색, 문맥=회색
COL = {"남색": "#051C2C", "파랑": "#2251FF", "하늘색": "#00A9F4",
       "회색": "#757575", "연회색": "#B3B3B3"}
FIG_W = 10.0     # 캔버스 폭은 오늘 그림 모두 이 값으로 고정하고 높이만 그림마다 조절한다


def datefmt(ax, fmt="%m-%d", n=5):
    """시간 축 눈금 개수와 표기를 잡아 날짜 라벨이 겹치지 않게 한다"""
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=n))
    ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))


def rmse(a, b):
    """두 계열의 평균제곱근오차 — ML 수업에서 쓰던 그 지표 그대로"""
    return float(np.sqrt(np.mean((np.asarray(a, dtype=float) - np.asarray(b, dtype=float)) ** 2)))


def mae(a, b):
    return float(np.mean(np.abs(np.asarray(a, dtype=float) - np.asarray(b, dtype=float))))

print("사용 폰트:", FONT)

In [ ]:
#@title [채점] 이 셀을 먼저 실행하세요 (한 번이면 됩니다)
#   미션의 「PROMPT 셀」을 실행하면 그 셀의 출력과 그림을 붙잡아 실행 결과에 채점 결과를 찍습니다.
#   통과하면 그 자리에서 완성본 풀이가 열리고, 아니면 항목별 힌트와 튜터에 보낼 질문이 나옵니다.
#   읽지 않고 실행만 해도 됩니다 — 기대값과 풀이는 이 셀에만 있고 「PROMPT 셀」에는 없습니다.
import base64
import glob
import io
import json
import re
import shutil
import subprocess
import sys

import matplotlib.font_manager as fm      # 아래 폰트 자급이 쓴다
import matplotlib.pyplot as plt


def _font_bootstrap():
    """한글 폰트 자급 — 맨 위 설치·폰트 셀을 건너뛰어도 그래프의 한글이 깨지지 않게.

    LMS 는 이 셀만 자동 실행하므로 폰트 준비도 여기서 한 번 더 한다. 설치·폰트 셀과 같은
    규칙이라 둘 다 돌아도 결과가 같고, apt-get 이 없는 자리(로컬)에서는 곧바로 지나간다.
    설치한 ttf 는 `addfont` 로 폰트 목록에 직접 넣는다 — 넣지 않으면 목록이 옛 상태라
    한글이 □ 로 깨진다(2026-09-09 Colab 실사고).
    """
    try:
        cands = ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"]
        installed = {f.name for f in fm.fontManager.ttflist}
        if not any(n in installed for n in cands) and shutil.which("apt-get"):
            subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"],
                           check=False, timeout=120)
        # 설치는 파일만 만든다 — 이미 떠 있는 폰트 목록에 addfont 로 넣어야 이름이 보인다.
        # (설치를 건너뛴 자리도 나눔 ttf 가 이미 있으면 이 줄들만으로 잡힌다.)
        for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):
            try: fm.fontManager.addfont(_p)
            except Exception: pass
        try: fm._get_font.cache_clear()
        except Exception: pass
        installed = {f.name for f in fm.fontManager.ttflist}
        _font = next((n for n in cands if n in installed), "DejaVu Sans")
        plt.rcParams["font.family"] = _font
        plt.rcParams["axes.unicode_minus"] = _font in {"Pretendard", "DejaVu Sans"}
    except Exception:
        print("한글 폰트를 못 찾아 그래프 글자가 깨질 수 있습니다 —"
              " 맨 위 설치·폰트 셀을 실행하세요")


_font_bootstrap()


def _dec_json(s):                          # base64(JSON) → 값. 리스트는 원래 튜플이라 되돌립니다
    def _t(v):
        return ({k: _t(x) for k, x in v.items()} if isinstance(v, dict)
                else tuple(_t(x) for x in v) if isinstance(v, list) else v)
    return _t(json.loads(base64.b64decode(s).decode("utf-8")))


# 채점 기준 — base64(JSON), 통과 판정에만 쓴다
_ANS = _dec_json(
           "eyJtMSI6eyJ2YWx1ZXMiOnsi7ZWY66OoIO2VqeqzhCDqs4Tsl7Qg66CI67KoIOywqOu2hCDtmp/siJggZCI6WzAuNSwxLjUsIu2aj"
           "CJdLCLtlZjro6gg7ZWp6rOEIOqzhOyXtCBBSUMiOls3NDU5LjksNzQ2MS45XX0sInBsb3QiOnsia2luZCI6ImJhciIsIm4iOjIxLC"
           "J5bWF4IjpbMC4wNSwwLjI1XSwi6riw7KSA7ISgX3kiOlswLjA5LDAuMTJdLCLquLDspIDshKBf7IiYIjoyLCLstpXrnbzrsqgiOnR"
           "ydWV9fSwibTIiOnsidmFsdWVzIjp7IuqzteyLnSDssKjsiJggQUlDIjpbMjUyMDguMywyNTIxMC4zXSwiMeychOyZgCAy7JyE7J2Y"
           "IEFJQyDssKjsnbQiOlsxLjMsMi4zXX0sInBsb3QiOnsia2luZCI6ImJhciIsIm4iOjYsIuy2leudvOuyqCI6dHJ1ZX0sIuywqOyIm"
           "F/tkZzquLAiOiIoMSwwLDEpIn19")
_REVEAL = {                                # 완성본 풀이 — base64, 통과할 때만 디코드해 인쇄합니다
    "m1": ("8J+UkyDrr7jshZggMSDsmYTshLHrs7gg4oCUIEFJIOy9lOuTnCDtkZzsi5zshKAg7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlaAg7L2U65Oc64qUIOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg"
           "7L2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg7J2066+4IOunjOuTpOyWtOyguCDsnojripQgZGYsIGRhaWx5X2MsIGF1dG9fYXJpbWEg66W8IOq3uOuMgOuhnCDsk7Dqs6Ag64uk"
           "7IucIOunjOuTpOyngOuKlCDrp5DslYQg7KSYIOKAlCBkYWlseV9jIOuKlCDsm5Drs7gg7Iuc6rCEIOqzhOyXtOydhCDtlZjro6gg7ZWp6rOE66GcIOynkeqzhO2VnAogICAgMzY17J28IOydvCDsiJjs"
           "mpTslbwuIGRhaWx5X2Mg7JeQIGF1dG9fYXJpbWEg66W8IOyggeyaqe2VtCDspJguIOqzhOygiCDssKjsiJjripQg6rOg66W07KeAIOunkOqzoChzZWFzb25hbD1GYWxzZSkgQUlDIOulvCDquLDspIDs"
           "nLzroZwg64uo6rOE67OEIO2DkOyDiQogICAgKHN0ZXB3aXNlPVRydWUp7Jy866GcIOu5hOqzhOygiCDssKjsiJjrp4wg6rOo6528IOykmCDigJQg7KCE7IiYIO2DkOyDieydgCDsk7Dsp4Ag66eQ6rOg"
           "LiDqs6Drpbgg7LCo7IiYIChwLCBkLCBxKSDsnZgg6rCA7Jq0642wIOqwkuydhCAi7ZWY66OoIO2VqeqzhCDqs4Tsl7Qg66CI67KoIOywqOu2hAogICAg7Zqf7IiYIGQgPSAo7Iir7J6QKSDtmowiIOuh"
           "nCDtlZwg7ZaJLCAi7ZWY66OoIO2VqeqzhCDqs4Tsl7QgQUlDID0gKOyIq+yekCkiIOulvCDshozsiJjsoJAg7JWE656YIDLsnpDrpqzroZwg7ZWcIO2WiSBwcmludCDtlbQg7KSYIOKAlCDsnbQg65GQ"
           "IO2WieydgCDshJzroZwg64uk66W4CiAgICDtlonsnbTslrTslbwg7ZWY6rOgIOuLpOuluCDsiKvsnpDqsIAg6rCZ7J2AIO2WieyXkCDshJ7snbTrqbQg7JWIIOuPvC4g6re4IOuLpOydjCBkYWlseV9j"
           "IOyXkOyEnCDtlZjro6gg7KCEIOqwkuydhCDruoAg6rOE7Je07J2EIOunjOuTpOqzoCDslZ7snZgg6rKw7Lih7J2EIOygnOqxsO2VnCDrkqQg7Iuc7LCoCiAgICAxIOu2gO2EsCAyMSDquYzsp4DsnZgg"
           "7J6Q6riw7IOB6rSA7J2EIOq1rO2VtCDspJguIDEuOTYg7J2EIOq3uCDqs4Tsl7Qg6ri47J207J2YIOygnOqzseq3vOycvOuhnCDrgpjriIgg6rCS7J20IOqyveqzhOyEoOydtOqzoCwg7LC46rOg66Gc"
           "IOyEoO2DneuQnCDssKjsiJjsmYAg6rOE7KCIIOywqOyImOyZgCDqsr3qs4TshKDqs7wKICAgIOyLnOywqCA3IMK3IDE0IMK3IDIxIOydmCDsnpDquLDsg4HqtIDsnYQg65Ox7Zi4IOyXhuydtCDtlZwg"
           "7ZaJ7JeQIHByaW50IO2VtCDspJguIOq3uOuemO2UhOuKlCDrlLEg7ZWcIOyepeunjCDigJQg7Iuc7LCoIDEg67aA7YSwIDIxIOq5jOyngOydmCDsnpDquLDsg4HqtIDsnYQg66eJ64yAIDIx6rCc66Gc"
           "CiAgICDrsLDsuZjtlZjqs6AgNyDsnZgg67Cw7IiY7J24IOyLnOywqOunjCDsg4nsnYQg64us66as7ZWcIOuSpCwg6rK96rOE7ISg7J2YIO2UjOufrOyKpOyZgCDrp4jsnbTrhIjsiqQg6rCS7JeQIOqw"
           "gOuhnCDsoJDshKDsnYQg7ZWY64KY7JSpIOq4i+qzoCAwIOyXkOuPhCDqsIDroZzshKDsnYQg7ZWY64KYIOq4i+qzoCDqsIDroZzstpXqs7wKICAgIOyEuOuhnOy2leyXkCDrnbzrsqjsnYQg67aA7Jes"
           "7ZW0IOykmC4KCiAgICBkYWlseV9jID0gZGZbImNvdW50Il0ucmVzYW1wbGUoIkQiKS5zdW0oKSAgICAgICAgICAjIOybkOuzuCDsi5zqsIQg6rOE7Je07J2YIO2VmOujqCDtlanqs4QsIDM2NeydvAog"
           "ICAgYXV0byA9IGF1dG9fYXJpbWEoZGFpbHlfYywgc2Vhc29uYWw9RmFsc2UsICAgICAgICAgICAgICAgICAgIyDqs4TsoIgg7LCo7IiY64qUIOydtOuyiCDtg5Dsg4nsl5DshJwg6rOg66W07KeAIOyV"
           "iuuKlOuLpAogICAgICAgICAgICAgICAgICAgICAgaW5mb3JtYXRpb25fY3JpdGVyaW9uPSJhaWMiLCBzdGVwd2lzZT1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgc3VwcHJlc3Nfd2FybmluZ3M9"
           "VHJ1ZSwgZXJyb3JfYWN0aW9uPSJpZ25vcmUiKQogICAgcCwgZCwgcSA9IGF1dG8ub3JkZXIKICAgIHByaW50KGYi7ZWY66OoIO2VqeqzhCDqs4Tsl7Qg66CI67KoIOywqOu2hCDtmp/siJggZCA9IHtk"
           "fSDtmowiKSAgICAgIyDssYTsoJDsnbQg7J2964qUIO2WiSAxIOKAlCDri6Trpbgg7Iir7J6Q66W8IOyEnuyngCDslYrripTri6QKICAgIHByaW50KGYi7ZWY66OoIO2VqeqzhCDqs4Tsl7QgQUlDID0g"
           "e2F1dG8uYWljKCk6LC4yZn0iKSAgICAgICMg7LGE7KCQ7J20IOydveuKlCDtlokgMgoKICAgIGRhaWx5X2RpZmYgPSBkYWlseV9jLmRpZmYoKS5kcm9wbmEoKSAgICAgICAgICAgICAgICAgICAgICAg"
           "IyDtlZjro6gg7KCEIOqwkuydhCDruoAg6rOE7Je0LCAzNjTsnbwKICAgIHIgPSBhY2YoZGFpbHlfZGlmZiwgbmxhZ3M9MjEpWzE6XSAgICAgICAgICAgICAgICAgICAgICAgICAgIyBbMTpdIOuhnCDr"
           "ipggMSDsnbgg7Iuc7LCoIDAg7J2EIOygnOqxsO2VnOuLpAogICAgYmFuZCA9IDEuOTYgLyBucC5zcXJ0KGxlbihkYWlseV9kaWZmKSkgICAgICAgICAgICAgICAgICAgICAjIOyekOujjCDsiJjqsIAg"
           "66eO7JWE7KeA66m0IOyigeyVhOyngOuKlCDqsr3qs4TshKAKICAgIHByaW50KGYi7LC46rOgIOKAlCDshKDtg53rkJwg7LCo7IiYIHthdXRvLm9yZGVyfSDCtyDqs4TsoIgg7LCo7IiYIHthdXRvLnNl"
           "YXNvbmFsX29yZGVyfSIKICAgICAgICAgIGYiIMK3IOqyveqzhOyEoCDCsXtiYW5kOi40Zn0gwrcg7Iuc7LCoIDcge3JbNl06Ky40Zn0gwrcg7Iuc7LCoIDE0IHtyWzEzXTorLjRmfSDCtyDsi5zssKgg"
           "MjEge3JbMjBdOisuNGZ9IikKCiAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KEZJR19XICogMC44LCAzLjQpKQogICAgY29sb3JzID0gW0NPTFsi7ZWY64qY7IOJIl0gaWYgayAlIDcg"
           "PT0gMCBlbHNlIENPTFsi7Jew7ZqM7IOJIl0gZm9yIGsgaW4gcmFuZ2UoMSwgMjIpXQogICAgYXguYmFyKHJhbmdlKDEsIDIyKSwgciwgY29sb3I9Y29sb3JzLCB3aWR0aD0wLjgpICAgICAgICMg7JqU"
           "7J28IOyLnOywqCA3IMK3IDE0IMK3IDIxIOunjCDsg4nsnLzroZwg6rWs67aE7ZWc64ukCiAgICBheC5heGhsaW5lKGJhbmQsIGNvbG9yPUNPTFsi7ZqM7IOJIl0sIGxzPSI6IikKICAgIGF4LmF4aGxp"
           "bmUoLWJhbmQsIGNvbG9yPUNPTFsi7ZqM7IOJIl0sIGxzPSI6IikKICAgIGF4LmF4aGxpbmUoMCwgY29sb3I9Q09MWyLrgqjsg4kiXSwgbHc9MC44KQogICAgYXguc2V0X3hsYWJlbCgi7Iuc7LCoICjs"
           "nbwpIikKICAgIGF4LnNldF95bGFiZWwoIuyekOq4sOyDgeq0gCIpCiAgICBwbHQuc2hvdygpCgrtlbTshJ0KCi0g7J6Q64+ZIOywqOyImCDtg5Dsg4nsnbQg6rOg66W4IOqyg+ydgCAoMSwgMSwgMykg"
           "7J206rOgIEFJQyDripQgNyw0NjAuOTQg7J6F64uI64ukLiDslZ4g6rSE7Zi464qUIOu5hOqzhOygiCDssKjsiJjroZwg7J6Q6riw7ZqM6reAIO2VrSAx6rCcLCDroIjrsqgg7LCo67aEIDHtmowsIOyd"
           "tOuPme2Pieq3oCDtla0gM+qwnOyeheuLiOuLpC4g6rCA7Jq0642wIOqwkuydtCAxIOydtOudvCDroIjrsqgg7LCo67aEIO2an+yImCBkIOuKlCAxIO2ajOyeheuLiOuLpC4KLSDqs4TsoIgg7LCo7IiY"
           "64qUICgwLCAwLCAwLCAwKSDsnLzroZwg7Lac66Cl65Cp64uI64ukLiBgc2Vhc29uYWw9RmFsc2VgIOuhnCDtmLjstpztlbQg7KO86riw66W8IOyEoOyWuO2VmOyngCDslYrslZjsnLzrr4DroZwg66eI"
           "7KeA66eJIOqwkuyduCDso7zquLDquYzsp4AgMCDsnoXri4jri6QuIOyalOydvCDso7zquLDrpbwg7ZuE67O07JeQIOuRmOyngOuKlCDshYvsp7gg64Kg7JeQIOuLpOujueuLiOuLpC4KLSDqt7jrprzs"
           "l5DshJwg7JqU7J28IOyLnOywqCjtlZjripjsg4kp66W8IOuztOuptCDsi5zssKggNyDsnbQg4oiSMC4wMzUyLCDsi5zssKggMTQg6rCAICswLjAzODAg7Jy866GcIOqyveqzhOyEoCDCsTAuMTAyNyDs"
           "lYjsqr3snbTqs6Ag7Iuc7LCoIDIxIOunjCArMC4xMDU5IOuhnCDqsqjsmrAg6rK96rOE7ISg7JeQIOuLv+yKteuLiOuLpC4g7JqU7J28IOq3nOy5meydtCDrmpzroLftlZjri6TrqbQg7Iuc7LCoIDcg"
           "wrcgMTQgwrcgMjEg7J20IO2VqOq7mCDsu6TsoLjslbwg7ZWY66+A66GcLCDsmpTsnbwg7Iuc7LCoIOyFiyDqsIDsmrTrjbAg7ZWY64KY66eMIOqyveqzhOyEoCDrsJTquaXsnbgg6rKD7J2AIOyasOyX"
           "sOycvOuhnCDrs7TripQg7Y647J20IOyekOyXsOyKpOufveyKteuLiOuLpC4KLSDsnbQg66+47IWY7J2YIOqysOuhoOydgCDssKjsiJjqsIAg7JWE64uZ64uI64ukLiDsnbQg66qo7ZiV7J20IOyCsOy2"
           "nO2VmOuKlCDsmIjsuKEgMeqwnOqwgCDjgIzqt7jrgqAg7ZWY66OoIO2VqeqzhOOAjSDrnbzripQg7KCQ7J6F64uI64ukLiDrsLDssKjtjIDsnYAgMTY47Iuc6rCE7J2EIOyalOyyre2WiOuKlOuNsCDs"
           "nbQg66qo7ZiV7J2AIDfsnbzsuZgg6rCSIDfqsJzrpbwg7KCc6rO17ZWp64uI64ukLiDsg4jrsr0gNOyLnOyXkCDtirjrn63snYQg66qHIOuMgCDsmrTtlontlaDsp4AsIDE47IucIOygleygkOyXkCDr"
           "qocg64yA66W8IOyxhOybjCDrkZjsp4Drpbwg7ZWY66OoIO2VqeqzhCDtlZjrgpjroZzripQg6rKw7KCV7ZWgIOyImCDsl4bsirXri4jri6QuCi0g7ZWY66OoIO2VqeqzhOuhnCDrjIDsi6DtlZjripQg"
           "6ri47J2AIOyXrOq4sOyEnCDrsLDsoJzrkKnri4jri6QuIOuLpOydjCDrr7jshZjsnYAg7Iuc6rCEIOuLqOychCDqs4Tsl7TroZwg6rCR64uI64ukLgoK7L2U65Oc6rCAIOuLteydhCDrp4zrk5zripQg"
           "67Cp7IudIOKAlCBgc2Vhc29uYWxgIOyduOyekCDtlZjrgpjqsIAg44CM7KO86riw66W8IOyEoOyWuO2VoCDqsoPsnbjqsIDjgI0g66W8IOqysOygle2VmOqzoCwgYEZhbHNlYCDrqbQg7YOQ7IOJ7J20"
           "IOu5hOqzhOygiCDssKjsiJggcCDCtyBkIMK3IHEg66eMIOqzoOumheuLiOuLpC4g6re466CH6rKMIOqzoOuluCBkIOqwgCAxIOydtOudvOuKlCDqsoMsIOq3uOumrOqzoCDsmIjsuKEgMeqwnOydmCDr"
           "i6jsnITqsIAg7ZWY66Oo652864qUIOqyg+ydtCDsnbQg66+47IWY7J2YIOuRkCDri7XsnoXri4jri6Qu"),
    "m2": ("8J+UkyDrr7jshZggMiDsmYTshLHrs7gg4oCUIO2VnCDqsIDsp4Ag6rCS7J20IOy9lOuTnCAz6rOz7JeQIOygge2YgCDsnojsl4jsirXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDtl"
           "ITroaztlITtirgg7JiICgogICAg7KCE7J6EIOuLtOuLueyekCDsvZTrk5zsl5DshJwg6rOg7LmgIOqzs+ydgCAxIOydtCDsoIHtnowgM+qzs+ydtOyVvCDigJQgU0FSSU1BWCDrpbwg7Zi47Lac7Z"
           "WY64qUIOy9lOuTnCDtloksIOywqOyImOulvCDstpzroKXtlZjripQgcHJpbnQsIOunieuMgCDrnbzrsqjsnYQg66eM65Oc64qUIOusuOyekOyXtC4g67mE6rOE7KCIIOywqOu2hAogICAg6rCS7Je"
           "QIDEg7J2EIOyggeyWtCDrkZDsl4jripTrjbAg6re4IOqwkuydgCDqs6DsoJXqsJIgRF9MRVZFTCDsnbTrnbwg6re464yA66GcIOyNqOyVvCDtlbQg4oCUIOqzhOygiOywqOu2hOydhCDsnbTrr7gg"
           "7KCB7Jqp7ZWcIOqzhOyXtOyXkCDroIjrsqgg7LCo67aE6rmM7KeAIOuNlO2VmOuptCDtlYTsmpQg7J207IOB7Jy866GcIOywqOu2hO2VnAogICAg6rOE7Je0KOqzvOywqOu2hCnsnbQg65CY6rGw6"
           "5OgLiBwcmludCDsmYAg652867KoIOusuOyekOyXtOydmCAxIOuPhCBEX0xFVkVMIOuhnCDrsJTqv5Qg7KSYLiDrgpjrqLjsp4DripQg6re464yA66GcIOuRkOqzoCDigJQgUF9HUklEIOyZgCBRX0"
           "dSSUQg7J2YIHAgwrcgcSDrpbwg7Ked7KeA7Ja0CiAgICA27KGw7ZWp7J2EIO2VmeyKtSDqtazqsIQgeV90cmFpbiDsl5Ag7KCB7ZWp7ZWY6rOgLCDqs4TsoIgg7LCo7IiY64qUICgwLCBEX1NFQVN"
           "PTkFMLCAwLCBNKSDroZwg6rOg7KCV7ZWY6rOgLCDsmLXshZjsnYAgdHJlbmQg7JeG7J2M7JeQIOygleyDgeyEscK36rCA7Jet7ISxIOqwleygnOuhnCDthrXsnbztlZjrqbQg64+8LgogICAg7KGw"
           "7ZWp66eI64ukIHAgwrcgcSDCtyBBSUMg66W8IO2VnCDtlonsnLzroZwg66qo7JWEIEFJQyDsmKTrpoTssKjsiJzsnLzroZwg7KCV66Cs7ZWcIO2RnOulvCDstpzroKXtlZjqs6AsIOunqCDsnIQg7"
           "KGw7ZWp7J2YIOyIq+yekCA36rCc66W8ICLqs7Xsi50g7LCo7IiYID0gKHAsZCxxKShQLEQsUSxtKSIKICAgIO2Yle2DnOuhnCDtlZwg7ZaJLCAi6rO17IudIOywqOyImCBBSUMgPSAo7Iir7J6QKS"
           "Ig66W8IOyGjOyImOygkCDslYTrnpggMuyekOumrOuhnCDtlZwg7ZaJLCDrp6gg7JyEIOuRkCDtlonsnZggQUlDIOulvCDruoAgIjHsnITsmYAgMuychOydmCBBSUMg7LCo7J20ID0gKOyIq+yekCk"
           "iIOulvCDtlZwg7ZaJCiAgICBwcmludCDtlbQg7KSYLiDqt7jrprzsnYAg7KCE7J6E7J6QIOqyg+qzvCDqsJnsnYAg7ZiV7YOc66GcIOuLpOyLnCDqt7jroKQg7KSYIOKAlCA27KGw7ZWp7J2YIEFJ"
           "QyDrpbwg6rCA66GcIOunieuMgCA26rCc66GcIOuwsOy5mO2VmOqzoCBBSUMg6rCAIOuCruydgCDqsoPsnbQg7JyE66GcIOyYpOqyjCDsoJXroKztlZjqs6Ag66eoIOychAogICAg7ZWY64KY66eMI"
           "OyDieydhCDri6zrpqztlZjqs6AsIOyEuOuhnCDriIjquIjsl5DripQg6rCBIOyhsO2VqeydmCDssKjsiJgg7ZGc6riw66W8IOyggeqzoCDqsIDroZzstpUg67KU7JyE64qUIDI1MDAwIOu2gO2EsC"
           "AyODAwMCDquYzsp4Ag65GQ6rOgIOqwgOuhnOy2leqzvCDshLjroZzstpXsl5Ag652867Ko7J2EIOu2gOyXrO2VtCDspJguCgogICAgcm93cyA9IFtdCiAgICBmb3IgcCwgcSBpbiBpdGVydG9vbHM"
           "ucHJvZHVjdChQX0dSSUQsIFFfR1JJRCk6CiAgICAgICAgX3IgPSBTQVJJTUFYKHlfdHJhaW4sIG9yZGVyPShwLCBEX0xFVkVMLCBxKSwgICAgICAgICAgIyDqsIDsmrTrjbAg6rCSID0g44CM7KSA"
           "67mEIOyFgOOAjeydmCDqs6DsoJXqsJIKICAgICAgICAgICAgICAgICAgICAgc2Vhc29uYWxfb3JkZXI9KDAsIERfU0VBU09OQUwsIDAsIE0pLAogICAgICAgICAgICAgICAgICAgICB0cmVuZD1Ob"
           "25lLCBlbmZvcmNlX3N0YXRpb25hcml0eT1UcnVlLAogICAgICAgICAgICAgICAgICAgICBlbmZvcmNlX2ludmVydGliaWxpdHk9VHJ1ZSkuZml0KGRpc3A9RmFsc2UpCiAgICAgICAgcm93cy5hcH"
           "BlbmQoeyJwIjogcCwgInEiOiBxLCAiQUlDIjogX3IuYWljfSkKCiAgICBncmlkID0gcGQuRGF0YUZyYW1lKHJvd3MpLnNvcnRfdmFsdWVzKCJBSUMiKS5yZXNldF9pbmRleChkcm9wPVRydWUpCiA"
           "gICBiZXN0ID0gZ3JpZC5pbG9jWzBdCiAgICBzZWNvbmQgPSBncmlkLmlsb2NbMV0gICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIOuwlOuhnCDri6TsnYwg7Iic7JyEIOKAlCDrkZAg"
           "6rCS7J2YIOywqOydtOqwgCDqsrDsoJXsnYQg6rCA66W464ukCiAgICBwcmludChncmlkLnJvdW5kKDIpLnRvX3N0cmluZyhpbmRleD1GYWxzZSkpCiAgICBwcmludChmIuqzteyLnSDssKjsiJggP"
           "SAoe2ludChiZXN0LnApfSx7RF9MRVZFTH0se2ludChiZXN0LnEpfSkiCiAgICAgICAgICBmIigwLHtEX1NFQVNPTkFMfSwwLHtNfSkiKQogICAgcHJpbnQoZiLqs7Xsi50g7LCo7IiYIEFJQyA9IH"
           "tiZXN0LkFJQzosLjJmfSIpCiAgICBwcmludChmIjHsnITsmYAgMuychOydmCBBSUMg7LCo7J20ID0ge3NlY29uZC5BSUMgLSBiZXN0LkFJQzosLjJmfSIpCgogICAgbGFiZWxzID0gW2YiKHtpbnQ"
           "oci5wKX0se0RfTEVWRUx9LHtpbnQoci5xKX0pKDAse0RfU0VBU09OQUx9LDAse019KSIKICAgICAgICAgICAgICBmb3IgciBpbiBncmlkLml0ZXJ0dXBsZXMoKV0KICAgIGZpZywgYXggPSBwbHQu"
           "c3VicGxvdHMoZmlnc2l6ZT0oRklHX1csIDQuNCkpCiAgICBjb2xvcnMgPSBbQ09MWyLtlZjripjsg4kiXSBpZiBpID09IDAgZWxzZSBDT0xbIuyXsO2ajOyDiSJdIGZvciBpIGluIHJhbmdlKGxlb"
           "ihncmlkKSldCiAgICBheC5iYXJoKHJhbmdlKGxlbihncmlkKSAtIDEsIC0xLCAtMSksIGdyaWRbIkFJQyJdLCBjb2xvcj1jb2xvcnMpCiAgICBheC5zZXRfeXRpY2tzKHJhbmdlKGxlbihncmlkKS"
           "AtIDEsIC0xLCAtMSkpCiAgICBheC5zZXRfeXRpY2tsYWJlbHMobGFiZWxzLCBmb250c2l6ZT0xMCkKICAgIGF4LnNldF94bGltKDI1MDAwLCAyODAwMCkKICAgIGF4LnNldF94bGFiZWwoIkFJQyA"
           "o64Ku7J2E7IiY66GdIOyii+ydjCkiKQogICAgYXguc2V0X3lsYWJlbCgi7LCo7IiYICjruYTqs4TsoIgpKOqzhOygiCkiKQogICAgcGx0LnNob3coKQoK7ZW07ISdCgotIOqzoOy5oCDqsJLsnYAg"
           "7ZWY64KY7J24642wIOygge2ejCDqs7PsnYAgM+qzs+ydtOyXiOyKteuLiOuLpCDigJQgYG9yZGVyPShwLCAxLCBxKWAg7J2YIOqwgOyatOuNsCDqsJIsIOywqOyImOulvCDstpzroKXtlZjripQgY"
           "HByaW50YCwg66eJ64yAIOudvOuyqCDrrLjsnpDsl7QuIOq3uCDqsJLsnYAg66qo7ZiV7J2YIO2Yle2DnOqwgCDslYTri4jrnbwg66qo7ZiV7J20IOyEpOuqhe2VoCDqs4Tsl7TsnYQg67CU6r+J64"
           "uI64ukLiDsoITsnoQg7L2U65Oc64qUIDbtlokg66qo65GQ7JeQIOugiOuyqCDssKjrtoTsnYQg7ZWcIOuyiOyUqSDrjZQg7ZaI7Jy864uIIDbtlonrgbzrpqzripQg67mE6rWQ6rCAIOuQqeuLiOu"
           "LpC4g64uk66eMIOq3uCDqs4Tsl7TsnYAg7LKr64KgIOynhOuLqOqzvCDri6Trpbgg6rOE7Je07J6F64uI64ukLiDsoIHtlakg7ZWcIOqzs+unjCDqs6DsuZjrqbQgQUlDIOuKlCDrp57sp4Drp4wg"
           "7ZmU66m07J2YIOywqOyImCDtkZzquLDsmYAg66eJ64yAIOudvOuyqOydtCBkID0gMSDsnbgg7LGE66GcIOuCqOqzoCwg7LGE7KCQ7J2AIOq3uCDrkZAg6rOz64+EIOu0heuLiOuLpC4KLSDqs6Dsu"
           "Zwg65Kk7J2YIDHsnITripQgU0FSSU1BKDEsMCwxKSgwLDEsMCwyNCkg7J206rOgIEFJQyDripQgMjUsMjA5LjM0IOyeheuLiOuLpC4g7J20IOywqOyImOulvCDrgrTsnbwg44CM7KSA67mEIOyFgO"
           "OAjeydtCDqt7jrjIDroZwg7J247Jqp7ZWp64uI64ukLgotIDLsnIQgKDIsMCwxKSgwLDEsMCwyNCkg64qUIDI1LDIxMS4xNiDsnLzroZwgMS44MyDssKjsnbTsnoXri4jri6Qo67CY7Jis66a8IOy"
           "ghCDqsJLsnLzroZwg6rOE7IKwKS4g6riw7KSA7J24IDIg7J207ZWY7JeQIOuTnOuKlCDssKjsnbTrnbwg44CM642UIOuCq+uLpOOAjSDqsIAg7JWE64uI6528IOOAjOq1rOuzhO2VoCDsiJgg7JeG"
           "64uk44CNIOuhnCDtlbTshJ3tlanri4jri6QuIOq3uOuemOyEnCDrqqjsiJjqsIAg7ZWY64KYIOyggeydgCDsqr3snYQg6rO17IudIOywqOyImOuhnCDsoJXtlojsirXri4jri6QuIDPsnIQgKDIsM"
           "CwwKSgwLDEsMCwyNCkg64qUIDI1LDIyMy42NSDroZwgMeychOyZgCAxNC4zMSDssKjsnbTsnbTqs6AsIOydtCDsoJXrj4TripQg6rWs67OE7ZWgIOyImCDsnojsirXri4jri6QuCi0g7KCE7J6EIO"
           "y9lOuTnOulvCDqt7jrjIDroZwg65GQ66m0IDHsnITqsIAgKDIsMSwxKSgwLDEsMCwyNCkg66GcIOuwlOuAjOqzoCBBSUMg64qUIDI1LDIxNy4zNCDroZwg64KY7Ji164uI64ukLiDsnbQg6rCS7J2"
           "AIGQgPSAwIO2RnOydmCAyNSwyMDkuMzQg7JmAIOuCmOuegO2eiCDrhpPsnYQg7IiYIOyXhuuKlCDsoJDsiJjsnoXri4jri6QuIOqwkuydtCDsobDquIgg7YuA66as64qUIOyCrOqzoOqwgCDslYTr"
           "i4jrnbwg7J207ZuEIOyImOyXheydtCDsnbjsmqntlaAg7LCo7IiY6rCAIOuwlOuAjOuKlCDsgqzqs6DsnoXri4jri6QuCgrsvZTrk5zqsIAg64u17J2EIOunjOuTnOuKlCDrsKnsi50g4oCUIGBvc"
           "mRlcmAg7Yqc7ZSM7J2YIOqwgOyatOuNsCDqsJIg7ZWY64KY6rCAIOOAjOyasOuPhOulvCDslrTripAg6rOE7Je07JeQ7IScIOqzhOyCsO2VoOyngOOAjSDrpbwg6rKw7KCV7ZWY6rOgLCDqt7gg6r"
           "CS7J20IO2RnCA27ZaJIOyghOu2gOyXkOyEnCDqsJnsnYQg65WM66eMIEFJQyDsl7TsnbQg7ISc66GcIOu5hOq1kO2VoCDsiJgg7J6I64qUIOygkOyImOqwgCDrkKnri4jri6QuIOq3uOumrOqzoCD"
           "qt7gg6rCS7J20IOyyq+uCoCDsp4Tri6jqs7wg6rCZ7JWE7JW8IO2RnOydmCAx7JyE6rCAIOuCtOydvCDsnbjsmqntlaAg7LCo7IiY6rCAIOuQqeuLiOuLpC4="),
}
# 항목별 힌트 — base64(JSON)
_HINT = _dec_json(
           "eyJtMSI6eyLtlZjro6gg7ZWp6rOEIOqzhOyXtCDroIjrsqgg7LCo67aEIO2an+yImCBkIjpbIu2DkOyDieydtCDqs6Drpbgg"
           "67mE6rOE7KCIIOywqOyImCAocCwgZCwgcSkg7J2YIOqwgOyatOuNsCDqsJLsnoXri4jri6Qg4oCUIOqysOqzvOydmCBgb3Jk"
           "ZXJgIOyXkCDqt7gg6rCSIDPqsJzqsIAg7J6I7Iq164uI64ukLiDqt7gg7JWe7JeQIGBzZWFzb25hbD1GYWxzZWAg66GcIO2Y"
           "uOy2nO2WiOuKlOyngCDrqLzsoIAg7ZmV7J247ZWY7IS47JqULiDqs4TsoIgg7KO86riw66W8IOyghOuLrO2VmOuptCDri6Tr"
           "pbgg66qo7ZiV7J20IOyEoO2DneuQmOyWtCDqsIDsmrTrjbAg6rCS64+EIOuLrOudvOyniCDsiJgg7J6I7Iq164uI64ukIiwi"
           "7J20IOy9lOuTnOqwgCDqs6Drpbgg7LCo7IiYIChwLCBkLCBxKSDsnZgg6rCA7Jq0642wIOqwkuydhCDslrTrlJTsl5DshJwg"
           "7J297JeI64qU6rCAIl0sIu2VmOujqCDtlanqs4Qg6rOE7Je0IEFJQyI6WyLslrTripAg6rOE7Je07JeQIOyWtOuKkCDsobDq"
           "sbTsnLzroZwg7KCB7Jqp7ZaI64qU7KeAIOywqOuhgOuhnCDtmZXsnbjtlZjshLjsmpQg4oCUIOybkOuzuCDsi5zqsIQg6rOE"
           "7Je07J2YIO2VmOujqCDtlanqs4QgMzY17J28IMK3IOqzhOygiCDssKjsiJjripQg6rOg66W07KeAIOyViuydjChgc2Vhc29u"
           "YWw9RmFsc2VgKSDCtyDsoJXrs7TquLDspIDsnYAgQUlDIMK3IOuLqOqzhOuzhCDtg5Dsg4nsnoXri4jri6QuIOuztOyglSDs"
           "iJjsmpTsnZgg7ZWY66OoIO2VqeqzhOuhnCDrsJTqvrjrqbQg64uk66W4IOqzhOyXtOydtOudvCDqsJLsnbQg7KCE67aAIOuL"
           "rOudvOyngOqzoCwg7KCE7IiYIO2DkOyDieydhCDsp4DsoJXtlZjrqbQgQUlDIOqwgCDri6zrnbzsoLgg6riw7KSAIOuylOyc"
           "hOulvCDrspfslrTrgqnri4jri6QiLCLsnbQg6rCS7J20IOyWtOuKkCDqs4Tsl7TsnZgg7Ja064qQIOyhsOqxtOyXkOyEnCDr"
           "gpjsmKgg7KCV67O06riw7KSA7J246rCAIl0sInBsb3QiOlsi66eJ64yA6rCAIO2VmOujqCDsoIQg6rCS7J2EIOu6gCDqs4Ts"
           "l7TsnZgg6rKD7J247KeAIMK3IOyLnOywqCAxIOu2gO2EsCAyMSDquYzsp4AgMjHqsJzsnbjsp4Agwrcg6rK96rOE7ISg7J2E"
           "IOygkOyEoCAy6rCc66GcIOq3uOyXiOuKlOyngCDCtyDrkZAg7LaV7JeQIOudvOuyqOydhCDrtoDsl6ztlojripTsp4Ag7LCo"
           "66GA66GcIO2ZleyduO2VmOyEuOyalC4g7LCo67aE7ZWY7KeAIOyViuydgCDqs4Tsl7TroZwg6re466as66m0IOunieuMgOqw"
           "gCDsoITrtoAg6rK96rOE7ISgIOuwlOq5peycvOuhnCDsu6Tsp4Dqs6AsIOyLnOywqCAwIOydmCDsnpDquLDsg4HqtIDsnYAg"
           "64qYIDEg7J206528IOq3uCDrp4nrjIDrpbwg64Ko6riw66m0IDIx6rCc6rCAIOyVhOuLiOqyjCDrkKnri4jri6QuIOqyveqz"
           "hOyEoOydgCAxLjk2IOydhCDsnpDro4wg7IiY7J2YIOygnOqzseq3vOycvOuhnCDrgpjriIgg6rCS7J2EIOychOyVhOuemOuh"
           "nCDtlZjrgpjslKkg6riL7Iq164uI64ukIiwi64K0IOq3uOumvOydmCDrp4nrjIDqsIAg7Ja064qQIOqzhOyXtOydmCDslrTr"
           "ipAg7Iuc7LCo67aA7YSw66W8IOq3uOumsCDqsoPsnbTqs6AsIOqyveqzhOyEoOydhCDrqocg6rCcIOq3uOyXiOuKlOqwgCJd"
           "LCJlcnJvciI6WyLsmKTrpZgg66y47J6lIOyyqyDtlonsnYQg6re464yA66GcIEFJIO2KnO2EsOyXkOqyjCDsoITtlZjqs6As"
           "IOOAjOykgOu5hCDshYDjgI3snbQg66eM65OgIOydtOumhChkZiDCtyBkYWlseV9jIMK3IGF1dG9fYXJpbWEp7J2AIOuLpOyL"
           "nCDrp4zrk6Tsp4Ag66eQ6rOgIOq3uOuMgOuhnCDsgqzsmqntlZjrnbzqs6Ag7KCB7Jy87IS47JqUIiwi7J20IOyYpOulmOqw"
           "gCDslrTripAg7L2U65OcIO2WieyXkOyEnCDsmZwg64Ks64qU7KeAIl19LCJtMiI6eyLqs7Xsi50g7LCo7IiYIEFJQyI6WyJB"
           "SSDsvZTrk5wg7ZGc7Iuc7ISgIOyVhOuemCBgb3JkZXI9YCDtipztlIzsnZggKirqsIDsmrTrjbAg6rCSKirsnYQg44CM7KSA"
           "67mEIOyFgOOAjeydtCDstpzroKXtlZwg6rOg7KCV6rCS6rO8IOu5hOq1kO2VmOyEuOyalCIsIuyyq+uCoOyXkOuKlCAyNOyL"
           "nOqwhCDqs4TsoIjssKjrtoQg7ZWcIOuyiOycvOuhnCDtj4nqt6Ag7IiY7KSA7J20IOydvOygle2VtOyhjOyKteuLiOuLpCjs"
           "nqzqsoDsoJUgQURGIOKIkjE3LjAgwrcgS1BTUyAwLjAwOCkuIOugiOuyqCDssKjrtoTsnYQg7ZWcIOuyiCDrjZQg7KCB7Jqp"
           "7ZWY66m0IO2VhOyalCDsnbTsg4HsnLzroZwg7LCo67aE7ZWcIOqzhOyXtOydtCDrkJjqs6AsIOq3uCDtkZzsnZggQUlDIOuK"
           "lCDssqvrgqAg7KeE64uo64yA66GcIOygge2Vqe2VnCDtkZzsnZggQUlDIOyZgCDrgpjrnoDtnogg64aT7J2EIOyImCDsl4bs"
           "irXri4jri6QiLCLsnbQg7ZGc7J2YIDbtlonsnbQg66qo65GQIOqwmeydgCDqs4Tsl7Tsl5DshJwg64KY7JioIOygkOyImOyd"
           "uOqwgCJdLCIx7JyE7JmAIDLsnITsnZggQUlDIOywqOydtCI6WyLjgIzqs7Xsi50g7LCo7IiYIEFJQ+OAjSDrj4Qg4p2MIOyd"
           "tOuptCDqt7gg7Lm47J2EIOqzoOy5mOuptCDsnbQg7Lm464+EIO2VqOq7mCDtkoDrpr3ri4jri6QuIOq3uCDsubjsnbQg4pyF"
           "IOyduOuNsCDsnbQg7Lm466eMIOKdjCDsnbTrqbQsIEFJQyDsmKTrpoTssKjsiJzsnLzroZwg7KCV66Cs7ZWcIO2RnOyXkOyE"
           "nCDrp6gg7JyEIO2WieqzvCDqt7jri6TsnYwg7ZaJ7J2YIEFJQyDrpbwg67qQ64qU7KeAIOuztOyEuOyalCIsIuuRkCDtlons"
           "nZgg7LCo67aEIOyhsOqxtOydtCDqsJnslYTslbwg6re4IOywqOydtOyXkCDrnLvsnbQg7J6I7Iq164uI64ukLiDqsIDsmrTr"
           "jbAg6rCS7J20IO2WieuniOuLpCDri6TrpbTrqbQg7J20IOyIq+yekOuPhCDri6Trpbgg6rOE7Je064G866as7J2YIOywqOyd"
           "tOqwgCDrkKnri4jri6QiLCLsnbQg7LCo7J2066W8IOyWtOuKkCDrkZAg7ZaJ7JeQ7IScIOu6kOqzoCwg65GQIO2WieydmCDs"
           "sKjrtoQg7KGw6rG07J20IOqwmeydgOqwgCJdLCJwbG90IjpbIuunieuMgOqwgCA26rCc7J247KeAIMK3IOuRkCDstpXsl5Ag"
           "652867Ko7J2EIOu2gOyXrO2WiOuKlOyngCDssKjroYDroZwg7ZmV7J247ZWY7IS47JqULiDtm4Trs7QgNuyhsO2VqeydhCDs"
           "oITrtoAg6re466Ck7JW8IO2VmOqzoCwg7IOB7JyEIOuqhyDtlonrp4wg64Ko6riw66m0IOqwnOyImOqwgCDslrTquIvrgqnr"
           "i4jri6QiLCLrgrQg6re466a87J2YIOunieuMgOqwgCDrqocg6rCc7J206rOgIOyWtOuKkCDsobDtlanrk6TsnYQg6re466aw"
           "IOqyg+yduOqwgCJdLCJlcnJvciI6WyLsmKTrpZgg66y47J6lIOyyqyDtlonsnYQg6re464yA66GcIEFJIO2KnO2EsOyXkOqy"
           "jCDsoITtlZjqs6AsIOOAjOykgOu5hCDshYDjgI3snbQg66eM65OgIOydtOumhCh5X3RyYWluIMK3IERfTEVWRUwgwrcgRF9T"
           "RUFTT05BTCDCtyBNIMK3IO2bhOuztCAy6rCcKeydgCDri6Tsi5wg66eM65Ok7KeAIOunkOqzoCDqt7jrjIDroZwg7IKs7Jqp"
           "7ZWY65286rOgIOyggeycvOyEuOyalCIsIuydtCDsmKTrpZjqsIAg7Ja064qQIOy9lOuTnCDtlonsl5DshJwg7JmcIOuCrOuK"
           "lOyngCJdLCLssKjsiJhf7ZGc6riwIjpbIuywqOyImOulvCDstpzroKXtlZjripQgYHByaW50YCDsmYAg66eJ64yAIOudvOuy"
           "qCDrrLjsnpDsl7TsnbQg7KCB7ZWp7JeQIOuEo+ydgCDqsJLqs7wg6rCZ7J2AIOqwkuydhCDsoIHripTsp4Ag67O07IS47JqU"
           "IOKAlCDsoIHtlanrp4wg6rOg7LmY6rOgIOuRkCDrrLjsnpDsl7TsnYQg6re464yA66GcIOuRkOuptCBBSUMg64qUIOunnuyV"
           "hOuPhCDtmZTrqbTsl5DripQg7JibIOywqOyImOqwgCDrgqjsirXri4jri6QiLCLtmZTrqbTsnZgg44CM6rO17IudIOywqOyI"
           "mOOAjSDtlonqs7wg66eJ64yAIOudvOuyqOydmCDssKjsiJgg7ZGc6riw6rCAIOyWtOuKkCDqsJLsnYQg7J297Ja0IOunjOuT"
           "pOyWtOyngOuKlOqwgCJdfX0=")
_ASK = {"m1": "full", "m2": "blank"}                       # 뒤로 갈수록 걷힙니다 — 미션 2 는 빈칸을 내가 채웁니다
_ASK_HEAD = {"full": "◼ 튜터에 그대로 보낼 질문 (코드는 고치지 말라고 명시합니다)",
             "blank": "◼ 튜터에 보낼 질문 (빈칸은 내가 채웁니다)",
             "bare": "◼ 튜터에 보낼 질문의 뼈대 (문장으로 만드는 것은 내 몫입니다)",
             "self": "◼ 스스로 점검할 항목의 구조 (프롬프트 명세는 내가 다시 씁니다)"}
_ASK_FORM = {"full": "내 PROMPT 는 「{prompt}」 였는데 「{name}」 채점이 ❌ 였습니다."
                     " AI 코드 표시선 아래 코드를 보고 「{ask}」 에 3~5문장으로 답해 주세요. 코드는 고치지 마세요.",
             "blank": "내 PROMPT 는 ____ 였는데 「{name}」 채점이 ❌ 였습니다. AI 코드 표시선 아래 코드에서"
                      " 「{ask}」 부터 짚고, 어느 코드 행을 무엇으로 고칠지 한 문장으로 답해 주세요.",
             "bare": "무엇을 : {ask} / 어디를 : {name} / 고치지 말 것 : 코드",
             "self": "무엇을 : {ask} / 어디를 : {name} / 다시 볼 곳 : AI 코드 표시선 아래 코드"}

_FIGS = []                                 # 「PROMPT 셀」이 만든 그림을 모으는 자리
_STATE = {"tag": None, "buf": None, "out": None, "ids0": set()}
_TRY = {}                                  # 미션마다 ❌ 가 몇 번째인지 — 2단 힌트가 이 수를 봅니다
_orig_figure = getattr(plt.figure, "_orig", plt.figure)   # 이 셀을 다시 실행해도 겹겹이 싸이지 않게


def _figure(*a, **k):
    f = _orig_figure(*a, **k)
    _FIGS.append(f)
    return f


_figure._orig = _orig_figure
plt.figure = _figure                       # subplots · pandas · seaborn 도 이 경로를 거칩니다


def _get(name):
    """이름이 살아 있으면 그 값, 없으면 None — 아직 만들지 않은 이름에도 채점이 죽지 않게"""
    return globals().get(name)


def _open_figs():
    """지금 열려 있는 그림들 — 래퍼를 거치지 않고 읽습니다"""
    try:
        return [plt._pylab_helpers.Gcf.figs[n].canvas.figure for n in plt.get_fignums()]
    except Exception:
        return []


def _nums(text):
    """출력 텍스트에서 숫자만 뽑습니다 (천 단위 쉼표 제거, 지수 표기 포함)"""
    picked = []
    for seg in re.findall(r"-?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?", text or ""):
        try:
            picked.append(float(seg.replace(",", "")))
        except ValueError:
            continue
    return picked


def _value_lines(txt, units):
    """값 채점이 볼 줄만 고릅니다 — 단위가 붙은 줄, 또는 「이름 = 값」 꼴의 줄.

    표를 통째로 띄우면 그 안에 정답이 우연히 섞여 있어, 「값을 골라 단위와 함께 출력한다」 는
    계약을 지키지 않고도 ✅ 가 열립니다. 그래서 단위를 달아 고른 줄이 하나라도 있으면 그
    줄들만 보고, 그런 줄이 아예 없을 때만 출력 전체를 봅니다.
    """
    picked = []
    for line in (txt or "").splitlines():
        if not line.strip():
            continue
        if "=" in line or any(unit and unit in line for unit in units):
            picked.append(line)
    return picked


def _label_lines(val_lines, label):
    """라벨이 든 값줄만 골라, 그 줄에서 라벨 뒤(「=」·「:」 뒤) 토막을 돌려줍니다.

    값줄의 숫자를 한 주머니에 담아 띠와 대조하면 **옆 칸의 값**이 이 칸을 헛통과시킵니다 —
    부호 칸 둘이 같은 띠일 때 한쪽에 찍힌 1 이 다른 칸까지 열어 주던 자리입니다. 그래서
    라벨이 든 줄이 있으면 그 줄의 라벨 뒤만 봅니다(없으면 부르는 쪽이 값줄 전체로 물러납니다).
    라벨 찾기는 정확히 같은 문자열 → 못 찾으면 공백·괄호를 걷어낸 꼴, 두 단입니다.
    """
    def _tighten(txt):                     # 공백과 괄호(안의 말까지) 를 걷어낸 글자 · 원래 자리
        keep, depth = [], 0
        for i, ch in enumerate(txt or ""):
            if ch in "([":
                depth += 1
            elif ch in ")]":
                depth = max(0, depth - 1)
            elif depth == 0 and not ch.isspace():
                keep.append(i)
        return "".join((txt or "")[i] for i in keep), keep

    tight_label = _tighten(label)[0]
    tails = []
    for line in val_lines:
        end = line.find(label) + len(label) if label and label in line else -1
        if end < 0 and tight_label:                       # 2단 — 공백·괄호를 걷어내고 다시 봅니다
            tight_line, keep = _tighten(line)
            at = tight_line.find(tight_label)
            end = (keep[at + len(tight_label) - 1] + 1) if at >= 0 else -1
        if end < 0:
            continue
        tail = line[end:]
        cut_at = [p for p in (tail.find("="), tail.find(":")) if p >= 0]
        tails.append(tail[min(cut_at) + 1:] if cut_at else tail)
    return tails


def _axes(fig):
    """Figure · Axes 어느 쪽을 받아도 축 목록으로 폅니다"""
    if fig is None:
        return []
    axs = getattr(fig, "axes", None)
    if isinstance(axs, list):
        return axs
    return [fig] if hasattr(fig, "get_xlabel") else []


def _seg_lines(ax):
    """`vlines` · `hlines` 로 그은 선 — Line2D 가 아니라 **LineCollection** 이라 `ax.lines` 에 없습니다.

    토막 하나를 `(가로값, 세로값)` 한 쌍으로 펴서 돌려줍니다. 이 길을 열어 두지 않으면
    `plt.vlines(...)` 로 화면에 분명히 그어 놓은 수직선이 「수직선이 아예 없다」 로 판정돼,
    학생이 고칠 곳 없는 결함을 고치러 가게 됩니다.
    """
    picked = []
    for c in getattr(ax, "collections", []):
        get_segs = getattr(c, "get_segments", None)      # LineCollection 에만 있는 메서드입니다
        if get_segs is None:                              # 산점도(PathCollection)·채움 영역(PolyCollection)은 선이 아닙니다
            continue
        try:
            segs = get_segs()
        except Exception:
            continue
        for seg in segs:
            xy = np.asarray(seg, dtype=float).reshape(-1, 2)
            if xy.shape[0] >= 2:
                picked.append((xy[:, 0], xy[:, 1]))
    return picked


def _as_float(val):
    """축에서 뽑은 좌표를 실수 배열로 — 시각으로 그은 선도 숫자로 폅니다.

    `axvline(시각)` 은 Timestamp 를 그대로 들고 있어 곧장 float 로 펴면 TypeError 가 납니다.
    그 한 줄 때문에 그림 채점이 통째로 「데이터를 뽑다 걸렸습니다」 로 끝나던 자리입니다.
    """
    arr = np.asarray(val)
    if arr.dtype == object:
        try:
            arr = np.asarray(pd.to_datetime(arr))
        except Exception:
            return np.asarray([], dtype=float)
    if arr.dtype.kind == "M":                  # 시각은 나노초 정수로 펴서 숫자처럼 다룹니다
        return arr.astype("datetime64[ns]").astype("int64").astype(float).reshape(-1)
    try:
        return np.asarray(arr, dtype=float).reshape(-1)
    except (TypeError, ValueError):
        return np.asarray([], dtype=float)


def _lines(ax):
    """축의 선을 곡선(점 10개 이상)과 수직선(x 가 같은 두 점)으로 갈라 돌려줍니다"""
    cur, vline = [], []
    cands = []
    for ln in ax.lines:                     # 선 하나가 깨져도 나머지는 살린다 — `axvline(x=날짜)` 처럼 x 가
        try:                                #  Timestamp 인 세로선(add_changepoints_to_plot)은 날짜를 숫자로 바꿔 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        cands.append((xd, yd))
    cands += _seg_lines(ax)                     # `plt.vlines` 로 그은 수직선도 여기서 합류합니다
    for xd, yd in cands:
        if xd.size != yd.size:
            continue
        if xd.size >= 10:
            cur.append((xd, yd))
        elif xd.size == 2 and np.isclose(xd[0], xd[1]):
            vline.append(float(xd[0]))
    return cur, vline


def _marker_lines(ax):
    """마커를 찍은 Line2D — `plot(x, y, "o")` 든 `plot(x, y, marker="o")` 든 점으로 셉니다.

    선으로 이으면서 점도 함께 찍은 선(`linestyle` 이 기본 실선)까지 세는 것은, 점추정을 그렇게
    찍은 그림이 「점추정 마커가 0개다」 로 ❌ 나던 자리이기 때문입니다. 점 계수는 linestyle 과
    무관하고, 마커를 달지 않은 선만 점에서 뺍니다.
    """
    picked = []
    for ln in ax.lines:
        marker = str(ln.get_marker()).strip()
        if marker in ("", "None", "none"):     # 마커 없는 선은 점이 아닙니다 — 선 모양은 보지 않습니다
            continue
        if marker in ("|", "_"):               # errorbar 의 capsize 캡은 점이 아닙니다 — 구간 끝을 그은 짧은 선입니다
            continue
        try:                                # 날짜 x 인 선 하나가 나머지 점까지 죽이지 않게 갈라 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        if xd.size and xd.size == yd.size:
            picked.append((ln, xd, yd))
    return picked


def _diagonal(ax, skip=(), xy=None):
    """y = x 인 기준선이 점이 놓인 범위를 가로지르는가 — 선이 있기만 하면 통과가 아닙니다"""
    drop_ids = {id(ln) for ln in skip}
    cands = [(np.asarray(ln.get_xdata(), dtype=float).reshape(-1),
           np.asarray(ln.get_ydata(), dtype=float).reshape(-1))
          for ln in ax.lines if id(ln) not in drop_ids]
    cands += _seg_lines(ax)                     # LineCollection 으로 그은 y = x 기준선도 셉니다
    for xd, yd in cands:
        if xd.size < 2 or xd.size != yd.size:
            continue
        if not (np.isfinite(xd).all() and np.isfinite(yd).all()):
            continue
        width = float(xd.max() - xd.min())
        if width <= 0:                                    # 수직선은 대각선이 아닙니다
            continue
        if not np.allclose(yd, xd, rtol=1e-3, atol=0.02 * width):
            continue                                   # 세로가 가로를 따라가지 않으면 넘어갑니다
        if xy is not None and getattr(xy, "size", 0):
            pt_span = float(xy[:, 0].max() - xy[:, 0].min())
            overlap = float(min(xd.max(), xy[:, 0].max()) - max(xd.min(), xy[:, 0].min()))
            if pt_span > 0 and overlap < 0.4 * pt_span:          # 점이 놓인 구간을 거의 덮지 못하는 토막 선
                continue
        return True
    return False


def _near(val, ref, factor=1.5):
    """실측값이 기준값의 배수 안에 드는가 — 눈금·표본 차이는 넘어갑니다"""
    try:
        val = float(val)
    except (TypeError, ValueError):
        return False
    if ref == 0:
        return abs(val) < 1e-9
    return val * ref > 0 and abs(ref) / factor <= abs(val) <= abs(ref) * factor


def _plot_gaps(ax, spec):
    """축 하나를 기준과 대조해 어긋난 곳을 문장 목록으로 돌려줍니다 (빈 목록이면 통과)"""
    gaps = []
    if spec.get("kind") == "line":
        cur, vline = _lines(ax)
        want = spec.get("n_series", len(cur))
        line_lo, line_hi = want if isinstance(want, tuple) else (want, want)   # 두 값이면 그 사이면 됩니다
        if not (line_lo <= len(cur) <= line_hi):
            ref_text = f"{line_lo}개" if line_lo == line_hi else f"{line_lo}개 이상 {line_hi}개 이하"
            gaps.append(f"곡선이 {len(cur)}개다 (기준 {ref_text})")
        if "n_points" in spec and cur:
            n_spec = spec["n_points"]              # 168시간치를 다 그렸는지 — 앞 몇 시간만 그린 그림을 거릅니다
            pt_lo, pt_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
            max_pts = max(int(x.size) for x, _ in cur)
            if not (pt_lo <= max_pts <= pt_hi):
                gaps.append(f"가장 긴 선의 점이 {max_pts}개다 (기준 {pt_lo}개 이상 {pt_hi}개 이하)")
        if cur:
            xd, yd = max(cur, key=lambda t: t[0].size)
            lo, hi = spec.get("x_range", (xd.min(), xd.max()))
            slack = 0.2 * (hi - lo)
            if not (xd.min() <= lo + slack and xd.max() >= hi - slack):
                gaps.append(f"가로축이 {xd.min():.3g}~{xd.max():.3g} 만 포함한다 (기준 {lo:g}~{hi:g})")
            if "peak_x" in spec:
                p0, p1 = spec["peak_x"]
                peak = float(xd[yd.argmax()])         # 로그우도면 봉우리가 그 자리에 섭니다
                trough = float(xd[yd.argmin()])         # NLL 로 그렸으면 골짜기가 그 자리입니다
                if not (p0 <= peak <= p1 or p0 <= trough <= p1):
                    gaps.append(f"곡선의 최댓값 위치가 {peak:.4f}, 최솟값 위치가 {trough:.4f} 다"
                                f" (기준 {p0}~{p1} — 둘 중 하나가 이 구간 안에 있어야 합니다)")
        if cur and ("봉우리_x" in spec or "골짜기_x" in spec):
            # 부호가 뜻을 가진 곡선(충격반응) — 위로 솟는 곡선과 아래로 꺼지는 곡선을 따로 봅니다.
            # 반응을 통째로 뒤집어 그리면 봉우리·골짜기의 자리는 그대로라 자리만 봐서는 걸러지지 않습니다
            if "봉우리_x" in spec:
                a0, a1 = spec["봉우리_x"]
                if not any(float(np.nanmax(y)) > 0 and a0 <= float(x[int(np.nanargmax(y))]) <= a1
                           for x, y in cur):
                    gaps.append(f"위로 솟아 {a0:g}~{a1:g} 시간 구간에서 최댓값에 이르는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
            if "골짜기_x" in spec:
                b0, b1 = spec["골짜기_x"]
                if not any(float(np.nanmin(y)) < 0 and b0 <= float(x[int(np.nanargmin(y))]) <= b1
                           for x, y in cur):
                    gaps.append(f"아래로 꺼져 {b0:g}~{b1:g} 시간 구간에서 최솟값에 이르는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
        if "vline_x" in spec:
            v0, v1 = spec["vline_x"]
            if not any(v0 <= v <= v1 for v in vline):
                gaps.append("기준 구간 안에 세로선이 없다"
                            + (f" (지금 {vline[0]:.4f})" if vline else " (세로선이 아예 없다)"))
    elif spec.get("kind") == "bar":
        heights = [float(p.get_height()) for p in ax.patches]
        if "n" in spec:
            n_spec = spec["n"]                          # 정수면 딱 그 개수, 두 값이면 그 사이(예: 시차 24~400)
            n_lo, n_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
            ref_text = f"{n_lo}개" if n_lo == n_hi else f"{n_lo}개 이상 {n_hi}개 이하"
            if not (n_lo <= len(heights) <= n_hi):
                gaps.append(f"막대가 {len(heights)}개다 (기준 {ref_text})")
        elif not heights:                              # 막대 수 기준이 없는 경우에도 빈 축은 통과시키지 않습니다
            gaps.append("막대가 하나도 없다 (이 축에서 막대를 찾지 못했습니다)")
        if heights and "ymax" in spec and not (spec["ymax"][0] <= max(heights) <= spec["ymax"][1]):
            gaps.append(f"가장 높은 막대가 {max(heights):.4f} 다 (기준 {spec['ymax']})")
    elif spec.get("kind") == "scatter":
        markers = _marker_lines(ax)                              # plot(x, y, "o") 로 찍은 점도 셉니다
        pt_sets = [np.asarray(c.get_offsets(), dtype=float).reshape(-1, 2)
              for c in ax.collections if not hasattr(c, "get_segments")]   # 선 컬렉션은 점으로 세지 않습니다
        pt_sets += [np.column_stack([xd, yd]) for _, xd, yd in markers]
        n_pts = int(sum(p.shape[0] for p in pt_sets))
        if "n_points" in spec:
            n_spec = spec["n_points"]
            n_lo, n_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
            if not (n_lo <= n_pts <= n_hi):
                gaps.append(f"점이 {n_pts}개다 (기준 {n_lo}개 — 가로축 눈금 하나에 점 하나,"
                            f" 강조로 덧찍는 점은 {n_hi - n_lo}개까지 넘어갑니다)")
        xy = np.vstack([p for p in pt_sets if p.size]) if any(p.size for p in pt_sets) else None
        if xy is not None:
            xy = xy[np.isfinite(xy).all(axis=1)]
        if xy is not None and xy.size:
            xs, ys = xy[:, 0], xy[:, 1]
            if "x_range" in spec:
                lo, hi = spec["x_range"]
                if not (_near(xs.min(), lo) and _near(xs.max(), hi)):
                    gaps.append(f"가로축 값이 {xs.min():.3g}~{xs.max():.3g} 다 (기준 {lo:g}~{hi:g})")
            if "y_range" in spec:
                lo, hi = spec["y_range"]
                if not (_near(ys.min(), lo) and _near(ys.max(), hi)):
                    gaps.append(f"세로축 값이 {ys.min():.4g}~{ys.max():.4g} 다 (기준 {lo:g}~{hi:g})")
            if "ymax" in spec and not (spec["ymax"][0] <= ys.max() <= spec["ymax"][1]):
                gaps.append(f"세로축 최댓값이 {ys.max():.4g} 다"
                            f" (기준 {spec['ymax'][0]:g}~{spec['ymax'][1]:g})")
            if "gap_min" in spec:
                spread = float(np.abs(ys - xs).max())
                if spread < spec["gap_min"]:
                    gaps.append(f"두 축의 값이 가장 크게 벌어진 곳도 {spread:.4f} 뿐이다"
                                f" (기준 {spec['gap_min']:g} 이상)")
        if spec.get("xscale") and ax.get_xscale() != spec["xscale"]:
            gaps.append(f"가로축이 {ax.get_xscale()} 눈금이다 (기준 {spec['xscale']} 눈금)")
        if spec.get("diag") and not _diagonal(ax, skip=[ln for ln, _, _ in markers], xy=xy):
            gaps.append("두 축의 값이 같은 점을 잇는 대각선(y = x)이 점이 놓인 구간에 없다")
    if "기준선_y" in spec:                       # 판정선처럼 가로 기준선을 요구하는 자리
        n_lo, n_hi = spec["기준선_y"]
        cands = [(np.asarray(ln.get_xdata(), dtype=float).reshape(-1),
               np.asarray(ln.get_ydata(), dtype=float).reshape(-1)) for ln in ax.lines]
        cands += _seg_lines(ax)                       # `hlines` 로 그은 기준선도 여기서 합류합니다
        h_levels = [float(yd[0]) for xd, yd in cands
              if yd.size >= 2 and np.isfinite(yd).all() and np.allclose(yd, yd[0])]
        ok_lines = [v for v in h_levels if n_lo <= abs(v) <= n_hi]
        need = int(spec.get("기준선_수", 1))
        if len(ok_lines) < need:
            gaps.append(f"높이 ±{n_lo:g}~{n_hi:g} 범위에 든 가로선이 {len(ok_lines)}개다"
                        f" (기준 {need}개 — 0 을 지나는 축선은 세지 않습니다)")
    if spec.get("띠"):                          # 위아래 경계 사이를 채운 면 — 경계선 두 개와 다릅니다
        faces = []
        for c in getattr(ax, "collections", []):
            if hasattr(c, "get_segments"):     # 선 컬렉션(LineCollection)은 면으로 세지 않습니다
                continue
            get_segs = getattr(c, "get_paths", None)
            if get_segs is None:
                continue
            try:
                faces += [p for p in get_segs()
                      if getattr(getattr(p, "vertices", None), "shape", (0,))[0] >= 4]
            except Exception:
                continue
        if not faces:
            gaps.append("위아래 경계 사이를 채운 영역이 없다"
                        " (경계선 두 개만으로는 예측구간 채움 영역이 되지 않습니다)")
    if spec.get("축라벨") and not (ax.get_xlabel().strip() and ax.get_ylabel().strip()):
        gaps.append("가로축·세로축 라벨 중 빈 것이 있다")
    return gaps


def _ref_plot_m1():
    """미션 1 의 참조 그림 — 「준비 셀」의 daily_c 로 차분한 일 계열 자기상관 막대를 그립니다"""
    x = _get("daily_c")
    if x is None or not hasattr(x, "diff"):
        return None
    diffed = x.diff().dropna()
    val = acf(np.asarray(diffed, dtype=float), nlags=21)[1:]     # [1:] 로 늘 1 인 시차 0 을 뗍니다
    band = 1.96 / np.sqrt(len(diffed))
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.4))
    a = f.add_subplot(111)
    colors = [COL["하늘색"] if k % 7 == 0 else COL["연회색"] for k in range(1, 22)]
    a.bar(np.arange(1, 22), val, color=colors, width=0.8)         # 요일 자리 7 · 14 · 21 을 색으로 구분합니다
    a.axhline(band, color=COL["회색"], ls=":")
    a.axhline(-band, color=COL["회색"], ls=":")
    a.axhline(0, color=COL["남색"], lw=0.8)
    a.set_xlabel("시차 (일)")
    a.set_ylabel("자기상관")
    return f


def _ref_plot_m2():
    """참조 그림 — 차분 조건을 고정한 6조합의 AIC 가로 막대"""
    tbl = [("(1,0,1)(0,1,0,24)", 25209.34), ("(2,0,1)(0,1,0,24)", 25211.16),   # 6조합을 다시 적합하는
          ("(2,0,0)(0,1,0,24)", 25223.65), ("(1,0,0)(0,1,0,24)", 25327.20),   # 대신, 참조 그림은 지난
          ("(0,0,1)(0,1,0,24)", 26294.07), ("(0,0,0)(0,1,0,24)", 27820.29)]   # 실행에서 얻은 값을 씁니다
    f = _orig_figure(figsize=(FIG_W, 4.4))
    a = f.add_subplot(111)
    keep = list(range(len(tbl) - 1, -1, -1))
    colors = [COL["하늘색"] if i == 0 else COL["연회색"] for i in range(len(tbl))]
    a.barh(keep, [v for _, v in tbl], color=colors)          # 가장 낮은 하나만 색으로 구분합니다
    a.set_yticks(keep)
    a.set_yticklabels([l for l, _ in tbl], fontsize=9)
    a.set_xlim(25000, 28000)
    a.set_xlabel("AIC (낮을수록 좋음)")
    a.set_ylabel("차수 (비계절)(계절)")
    return f


def auto_arima_like(y, seasonal=True, m=1, information_criterion="aic", **_):
    """pmdarima 를 못 쓸 때의 대체 — 차분 횟수를 검정으로 먼저 정하고 나머지만 훑는다.

    미션 1 「준비 셀」이 pmdarima 를 불러오지 못했을 때만 `auto_arima` 로 이 함수를 씁니다.
    pmdarima 와 다른 차수를 골라 AIC 가 기준 범위를 벗어나므로, 이 함수를 쓴 날은
    채점이 AIC 칸을 보지 않습니다(아래 `_SKIP`).
    """
    import itertools as _it
    import numpy as _np
    import pandas as _pd
    from statsmodels.tsa.seasonal import STL
    from statsmodels.tsa.stattools import kpss
    from statsmodels.tsa.statespace.sarimax import SARIMAX as _SARIMAX
    x, d = _np.asarray(y, dtype=float), 0
    while d < 2 and kpss(x, regression="c", nlags="auto")[1] < 0.05:
        x, d = _np.diff(x), d + 1              # 평균 수준이 일정하지 않으면(KPSS 기각) 한 번 더 차분한다
    use_seasonal = bool(seasonal) and m > 1
    D = 0
    if use_seasonal:
        _r = STL(_pd.Series(_np.asarray(y, dtype=float)), period=m, robust=True).fit()
        _strength = max(0.0, 1 - _r.resid.var() / (_r.seasonal + _r.resid).var())
        D = 1 if _strength >= 0.64 else 0            # 계절 강도가 0.64 이상일 때만 계절차분을 넣는다
    best_combo, seasonal_orders = None, (0, 1) if use_seasonal else (0,)
    for p, q, P, Q in _it.product((0, 1), (0, 1), seasonal_orders, seasonal_orders):
        try:
            _f = _SARIMAX(y, order=(p, d, q), seasonal_order=(P, D, Q, m if use_seasonal else 0),
                          trend=None, enforce_stationarity=True,
                          enforce_invertibility=True).fit(disp=False)
        except Exception:
            continue                          # 수렴하지 않는 조합은 후보에서 빠진다
        if best_combo is None or float(_f.aic) < best_combo[0]:
            best_combo = (float(_f.aic), (p, d, q), (P, D, Q, m if use_seasonal else 0))

    class _result:                                # pmdarima 결과와 같은 자리에서 읽히게
        def __init__(self, aic, order, seasonal_order):
            self._aic, self.order, self.seasonal_order = aic, order, seasonal_order

        def aic(self):
            return self._aic

    return _result(*best_combo)


# 채점하지 않는 칸 — (미션, 라벨) → 그 칸을 건너뛸 조건과 알림 문장
_SKIP = {("m1", "하루 합계 계열 AIC"): (
    lambda: "auto_arima_like" in str(_get("_search_backend") or ""),
    "pmdarima 를 설치하지 못해 대체 탐색을 썼습니다. 대체 탐색은 다른 차수를 골라 AIC 가"
    " 기준과 달라지므로 이 칸은 채점하지 않습니다")}


_REF = {"m1": _ref_plot_m1, "m2": _ref_plot_m2}   # 미션마다 참조 그림을 그리는 함수 하나


def _show(fig):
    """채점 훅 안에서 그림을 화면에 올립니다 (인라인 백엔드는 이 시점에 plt.show() 가 듣지 않습니다)"""
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        plt.show()


def _ref_show(fig, key):
    """내 그림과 참조 그림을 나란히 — 내 그림이 없으면 참조 그림도 띄우지 않습니다"""
    if not _axes(fig) or key not in _REF:
        return
    ref = None
    try:
        ref = _REF[key]()
        if ref is None:
            return
        panels = [(fig, "내 그림"), (ref, "참조 그림")]
        f = _orig_figure(figsize=(FIG_W, 3.4))
        for i, (g, title) in enumerate(panels):
            a = f.add_subplot(1, len(panels), i + 1)
            buf = io.BytesIO()
            (g if hasattr(g, "savefig") else g.figure).savefig(buf, format="png", dpi=110,
                                                               bbox_inches="tight")
            buf.seek(0)
            a.imshow(plt.imread(buf))
            a.axis("off")
            a.set_title(title, fontsize=11)
        _show(f)
        plt.close(f)
    except Exception:
        print("ⓘ 참조 그림을 나란히 놓지 못했습니다 — 위 채점 결과는 그대로 유효합니다")
    finally:
        if ref is not None:
            plt.close(ref)


def _one_line(e, n=120):
    """예외를 한 줄로 — 트레이스백 대신 학생이 읽을 문장 하나만 남깁니다"""
    txt = str(e).replace("\n", " ").strip()
    return (txt[:n] + "…") if len(txt) > n else txt


def _prompt1(key=None, n=70):
    """지금 셀의 PROMPT 입력란 첫 행 — 튜터에 보낼 질문에 그대로 싣습니다"""
    head_lines = [l.strip() for l in str(_get("PROMPT") or "").split("\n") if l.strip()]
    return (head_lines[0][:n] + "…") if head_lines else "____"


_PROMPT_MIN = 30
_PROMPT_PLACEHOLDERS = ("", "목표", "쓸 변수", "출력", "목표 —", "쓸 변수 —", "출력 —", "TODO",
                    "여기에 내 말로 적습니다")


def _strip(txt):
    """빈칸·줄표·가운뎃점을 떼어 낸 알맹이 — 자리표시자 대조는 이 꼴로 합니다"""
    return re.sub(r"[\s—·:,\-]+", "", str(txt or ""))


_PLACEHOLDERS_STRIPPED = {_strip(t) for t in _PROMPT_PLACEHOLDERS}


def _prompt_missing(key=None):
    """그 미션의 PROMPT 입력란이 아직 내 말로 채워지지 않았으면 사유 한 문장, 채워졌으면 None"""
    cell = "PROMPT"                              # 미션 1·2·3·과제 모두 PROMPT 입력란입니다
    try:
        txt = str(_ip.user_ns.get(cell, _get(cell)) or "").strip()
    except Exception:
        txt = str(_get(cell) or "").strip()
    stripped = _strip(txt)                          # 길이보다 먼저 본다 — 견본을 그대로 둔 자리가 있다
    if not stripped:
        return f"{cell} 입력란이 비어 있습니다"
    if stripped in _PLACEHOLDERS_STRIPPED:
        return f"{cell} 입력란이 자리표시자·견본 그대로입니다"
    if len(txt) < _PROMPT_MIN:
        return f"{cell} 입력란이 {len(txt)}자뿐입니다 (최소 {_PROMPT_MIN}자)"
    return None


def _reveal_show(txt):
    """완성본 코드와 해설 — **평문 그대로** 찍는다.

    LMS 렌더러 계약: 「🔓 정답이 열렸습니다 …」 줄부터 채점 표를 닫는 `─` 구분선 줄까지를
    출력 패널이 「🔓 완성본 풀이 보기」 상자로 접는다. 접힘의 표지가 이 **평문 마커**라
    HTML `details` 로 싣지 않는다 — 마커도 이 함수도 바꾸지 않는다.
    """
    print(txt)


def judge(key, text="", figs=(), error=None):
    """출력 텍스트와 그림을 기준과 대조해 채점 표를 찍습니다 — 통과하면 완성본 풀이를 엽니다"""
    spec = _ANS.get(key)
    if spec is None:                       # 기준이 아직 없는 미션은 조용히 넘어갑니다
        return None
    head_label = ("미션 " + key[1:]) if key[:1] == "m" and key[1:].isdigit() else key
    failed, passed = [], True
    print("\n" + "─" * 54)
    print(f"◼ {head_label} 채점 결과")
    reason = _prompt_missing(key)                     # 값·그래프보다 먼저 — 내 말로 시킨 자리가 있는가
    _locked = bool(reason)
    if reason and key != "m2":
        print(f"❌ {reason} — 짧게 시켜도 됩니다. 다만 무엇을 어떤 단위로 출력할지까지 적으면 한 번에 통과할 확률이 높습니다")
        print("   PROMPT 입력란을 채우기 전에는 값·그래프 채점도,"
              " 완성본 풀이도 열지 않습니다")
        print("─" * 54)
        return False
    elif reason:                                   # 함정 미션만은 전임자 코드의 진단이 먼저다
        print(f"ⓘ {reason} — 아래 채점 결과는 전임 담당자 코드의 진단입니다."
              " 고칠 지시를 PROMPT 에 쓰세요")
    if error is not None:                   # 오류가 나면 값·그래프 채점은 뜻이 없습니다
        print(f"❌ 실행 오류 — {type(error).__name__}: {_one_line(error)}")
        hint, question = _HINT.get(key, {}).get("error", ("", "이 오류가 어느 줄에서 왜 났는지"))
        if hint:
            print(f"   힌트 : {hint}")
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        print("  " + _ASK_FORM[form].format(name="실행 오류", prompt=_prompt1(key), ask=question))
        print("─" * 54)
        return False
    txt = text or ""
    if not txt.strip():
        print("❌ 출력이 없습니다 — 값을 print 하도록 프롬프트의 출력 형식에 적으세요")
    units = [str(ref[2]) for ref in spec.get("values", {}).values() if len(ref) > 2]
    val_lines = _value_lines(txt, units)                    # 단위·「=」 가 붙은 줄이 있으면 그 줄만 봅니다
    nums = _nums("\n".join(val_lines)) if val_lines else _nums(txt)
    table_flood = not val_lines and len(nums) > 30        # 골라낸 자리 없이 숫자만 잔뜩 = 표를 통째로 띄운 모양
    if table_flood:
        print(f"❌ 값 — 단위도 「이름 = 값」 도 없이 숫자가 {len(nums)}개 나왔습니다 — 표를"
              " 통째로 띄우면 어느 숫자가 답인지 채점이 고르지 못합니다."
              " 값은 「이름 = 숫자 단위」 한 행으로 출력하세요")
        nums = []
    for label, ref in spec.get("values", {}).items():
        skip_if, skip_msg = _SKIP.get((key, label), (None, ""))
        if skip_if is not None and skip_if():
            print(f"ⓘ {label} — {skip_msg}")
            continue
        lo, hi = float(ref[0]), float(ref[1])
        unit = ref[2] if len(ref) > 2 else ""
        label_head = ref[3] if len(ref) > 3 else ""    # 그 낱말이 적힌 줄에서만 숫자를 봅니다
        cands = nums
        fell_back = False                              # 라벨이 든 행이 없어 값줄 전체로 물러났는지
        if label_head and not table_flood:                   # 이름이 값을 가르는 자리 — 「확신」·「보류」
            cands = _nums("\n".join(line.split(label_head, 1)[1]      # 그 낱말 뒤쪽만 봅니다
                                  for line in (val_lines or txt.splitlines()) if label_head in line))
        elif nums:                                  # 이 라벨이 든 행의 「=」 뒤만 — 옆 칸과 안 섞이게
            tail = _label_lines(val_lines or txt.splitlines(), label)
            cands = _nums("\n".join(tail)) if tail else nums
            fell_back = not tail
        hits = [v for v in cands if lo <= v <= hi]
        if hits:
            print(f"✅ {label} — 출력에서 {hits[0]:,.6g} 를 찾았습니다")
            if unit and unit not in txt:
                print(f"   ⓘ 단위 「{unit}」 도 값과 함께 출력하면 보고서에 그대로 옮길 수 있습니다")
        elif table_flood:
            passed = False
            failed.append((label, label))
            print(f"❌ {label} — 「이름 = 숫자 {unit}」 한 행으로 골라 출력하면 채점이 읽습니다")
        else:
            passed = False
            failed.append((label, label))
            where = f"「{label_head}」 이 적힌 행에" if label_head else "출력에"
            _prev_fail = _TRY.get(key, 0)              # 결정 16 — 첫 ❌ 에는 기준 범위를 감춘다
            print(f"❌ {label} — 기준을 충족하는 숫자가 {where} 없습니다"
                  + (f" (기준 범위 {lo:g} ~ {hi:g})" if _prev_fail else ""))
            if fell_back:                            # 라벨 행을 못 찾아 값줄 전체를 본 자리
                print(f"   ⓘ 라벨 「{label}」 이 든 행을 찾지 못해 값줄 전체에서 찾았습니다"
                      " — 「이름 = 숫자 단위」 한 행으로 출력하면 그 행만 보고 채점합니다")
    plot_spec = spec.get("plot")
    if plot_spec:
        axs = [a for f in figs for a in _axes(f)]
        if not axs:
            passed = False
            failed.append(("그래프", "plot"))
            print("❌ 그래프 — 그림을 찾지 못했습니다 (그래프 한 장도 같은 프롬프트에 함께 시킵니다)")
        else:
            need_axes = int(plot_spec.get("n_axes", 1))   # 패널 두 장을 나란히 요구하는 기준이면 2 입니다
            ok_axes, gaps = 0, None
            for ax in axs:                 # 기준에 맞는 축이 필요한 수만큼 있으면 통과입니다
                try:
                    this_gaps = _plot_gaps(ax, plot_spec)
                except Exception as e:
                    this_gaps = [f"그림에서 데이터를 뽑다 걸렸습니다 ({type(e).__name__})"]
                if not this_gaps:
                    ok_axes += 1
                    if ok_axes >= need_axes:
                        break
                elif gaps is None or len(this_gaps) < len(gaps):
                    gaps = this_gaps
            if ok_axes < need_axes:
                passed = False
                failed.append(("그래프", "plot"))
                extra_msgs = [f"기준에 맞는 패널이 {ok_axes}개다 (기준 {need_axes}개)"] if need_axes > 1 else []
                print("❌ 그래프 — " + " · ".join(extra_msgs + (gaps or [])))
            else:
                print("✅ 그래프 — 기준대로 그렸습니다")
            _ref_show(figs[0] if figs else None, key)
    order_tag = spec.get("차수_표기")               # 적합만 고치고 출력 문자열을 그대로 두면 ✅ 가 열리던 자리
    if order_tag:
        def _tight(t):                              # 「(1, 0, 1)」 과 「(1,0,1)」 을 같은 표기로 본다
            return re.sub(r"\s+", "", str(t or ""))
        in_text = order_tag in _tight(txt)
        tick_txt = []
        for f in figs:
            for a in _axes(f):
                try:
                    tick_txt += [_tight(t.get_text()) for t in a.get_yticklabels()]
                except Exception:
                    continue
        in_ticks = any(order_tag in t for t in tick_txt)
        if in_text and in_ticks:
            print("✅ 차수 표기 — 「공식 차수」 행과 막대 라벨이 고친 차수를 적고 있습니다")
        else:
            passed = False
            failed.append(("차수 표기", "차수_표기"))
            missing = [w for w, ok in (("「공식 차수」 행", in_text), ("막대 라벨", in_ticks)) if not ok]
            print(f"❌ 차수 표기 — {' · '.join(missing)}에서 고친 차수 표기를 찾지 못했습니다")
    tbl = _HINT.get(key, {})
    attempt = _TRY[key] = _TRY.get(key, 0) + (1 if failed else 0)
    for name, hkey in failed:
        tiers = tbl.get(hkey, ("",))
        tier = tiers[:-1] if len(tiers) > 2 else tiers[:1]   # 2단 힌트는 마지막 원소가 튜터에 물을 것입니다
        hint = tier[min(attempt, len(tier)) - 1] if tier else ""
        if hint:
            print(f"   힌트({name}) : {hint}")
    if failed:
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        for name, hkey in failed:
            question = (tbl.get(hkey, ("", ""))[-1] if len(tbl.get(hkey, ("",))) > 1 else "") \
                or "이 항목이 왜 ❌ 로 남는지"
            print("  " + _ASK_FORM[form].format(name=name, prompt=_prompt1(key), ask=question))
    elif _locked:                                  # 값·그림은 통과했지만 내 말로 시킨 자리가 아직 비었습니다
        print("ⓘ 값과 그림은 채점 기준을 충족했습니다 — PROMPT 에 고칠 지시를 내 말로 쓰고"
              " 다시 실행하면 완성본 풀이가 열립니다")
    else:
        print("🔓 정답이 열렸습니다 — 아래 「🔓 완성본 풀이 보기」를 펼쳐 내 코드와 견줘 보세요")
        _reveal_show(base64.b64decode(_REVEAL[key]).decode("utf-8"))
    print("─" * 54)
    return passed and not _locked


class _Tee:
    """「PROMPT 셀」의 출력을 화면과 버퍼 양쪽으로 흘려보냅니다"""

    def __init__(self, orig, buf):
        self._o, self._b = orig, buf

    def write(self, s):
        self._b.write(s)
        return self._o.write(s)

    def flush(self):
        self._o.flush()

    def __getattr__(self, n):
        return getattr(self._o, n)


def _tag(src):
    """첫 줄이 `# 미션 N — 프롬프트` · `# 과제 — 프롬프트` 인 셀만 채점 대상입니다"""
    head_line = (src or "").split("\n")[0]
    m = re.match(r"\s*#\s*(미션\s*(\d+)|과제)\b", head_line)
    if not m or "프롬프트" not in head_line:                     # 「준비 셀」(`# 미션 N · 준비`)은 대상이 아닙니다
        return None
    return "m" + m.group(2) if m.group(2) else "과제"


def _pre(info=None):
    """「PROMPT 셀」면 출력 버퍼를 걸고 그림 목록을 비웁니다"""
    try:
        tag = _tag(getattr(info, "raw_cell", ""))
        _STATE["tag"] = tag
        if tag is None or tag not in _ANS:
            return
        _FIGS.clear()
        _STATE["ids0"] = {id(f) for f in _open_figs()}
        _STATE["buf"], _STATE["out"] = io.StringIO(), sys.stdout
        sys.stdout = _Tee(sys.stdout, _STATE["buf"])
    except Exception:
        pass


def _post(result=None):
    """「PROMPT 셀」이 끝나면 출력을 되돌리고 그 셀의 출력·그림으로 채점합니다"""
    tag, buf, out = _STATE.get("tag"), _STATE.get("buf"), _STATE.get("out")
    _STATE["tag"], _STATE["buf"], _STATE["out"] = None, None, None
    if out is not None:
        sys.stdout = out
    if tag is None or tag not in _ANS:
        return
    seen = {id(f) for f in _FIGS}
    figs_all = list(_FIGS) + [f for f in _open_figs()
                         if id(f) not in seen and id(f) not in _STATE.get("ids0", set())]
    txt = buf.getvalue() if buf is not None else ""
    last_val = getattr(result, "result", None)       # 셀 마지막 표현식의 값 — print 없이 표로 띄운 값도 읽습니다
    if last_val is not None:
        try:
            txt = txt + "\n" + str(last_val)
        except Exception:
            pass
    try:
        judge(tag, txt, figs_all, error=getattr(result, "error_in_exec", None))
    except Exception as e:
        print(f"(채점을 마치지 못했습니다 — {type(e).__name__}: {_one_line(e)})")


try:
    _ip = get_ipython()
    for _ev, _fn in (("pre_run_cell", _pre), ("post_run_cell", _post)):
        for _old in list(_ip.events.callbacks[_ev]):
            if getattr(_old, "__name__", "") == _fn.__name__:
                _ip.events.unregister(_ev, _old)     # 이 셀을 다시 실행해도 채점은 한 번만 찍힙니다
        _ip.events.register(_ev, _fn)
    print("채점 준비 완료 — 미션의 「PROMPT 셀」을 실행하면 실행 결과(출력)에 채점 결과가 찍힙니다")
except Exception:
    print("채점 준비 완료 (IPython 밖이라 훅 없이 judge() 만 올렸습니다)")

## 1. 하루 합계 계열의 검토

시간이 모자라면 미션 2 만 합니다. 미션 1 은 시간이 남을 때 합니다.

### 미션 1 [선택] — 하루 합계 365일 계열로 168시간을 대신할 수 있는가

#### 할 일 — 미션 1 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 시킵니다.
3. 셀을 클릭한 채 AI 튜터에게 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶.
4. 실행 결과에 출력되는 채점 결과를 봅니다. ❌ 면 힌트대로 PROMPT 를 고쳐 3번부터 다시 합니다.

**이 미션의 질문** — 배차팀이 요청한 168시간을 하루 합계 365일 계열로 대신할 수 있습니까?

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `daily_c` | 일 수요 — 원본 시간 계열의 하루 합계 365일, 단위 대/일 | 고정 |
| `auto_arima` | 자동 차수 탐색 함수 | 그대로 호출한다 |
| `df` · `df['count']` | 시간당 기록 표 전체와 원본 시간 계열, 단위 대 | 고정 |
| `acf` | 시차별 자기상관을 한 번에 반환하는 함수 | 그대로 호출한다 |

#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 모두 출력되게 합니다. 굵은 이름은 글자 그대로 사용합니다. `daily_c` 에 자동 차수 탐색을 한 번 적용하고, 1번과 2번은 그 결과에서 읽습니다.

1. **하루 합계 계열 레벨 차분 횟수 d** — 단위 회 · 정수 · 탐색이 고른 비계절 차수 (p, d, q) 의 가운데 값
2. **하루 합계 계열 AIC** — 단위 없는 점수 · 소수점 아래 2자리 · 그 모형의 AIC
3. **그래프 한 장** — 가로축은 시차 1일부터 21일까지, 세로축은 자기상관인 막대 그래프
   - 대상 계열 : `daily_c` 에서 하루 전 값을 뺀 계열(레벨 차분 1회)
   - 막대 : 시차 1 부터 21 까지 21개
   - 색 : 요일에 해당하는 시차 7 · 14 · 21 만 다른 색으로 강조합니다
   - 가로 점선 : 경계선 ±1.96/√T 를 가로축 전체에 걸쳐 위아래 하나씩 2개
   - 경계선 : 자기상관이 0 과 구별되는지 가르는 선. T 는 그 계열의 자료 수
   - 가로축 이름 : 「시차 (일)」 · 세로축 이름 : 「자기상관」
4. **참고 행** — 채점하지 않는 한 행 · 탐색이 고른 차수와 계절 차수, 경계선, 시차 7 · 14 · 21 의 자기상관을 등호 없이 적습니다. 퀴즈 두 문항이 이 행을 봅니다

**조건** — 이번 탐색은 계절 차수를 고르지 않습니다. `seasonal=False` 는 계절 주기를 선언하지 않겠다는 뜻입니다. 그러면 탐색은 비계절 차수 p · d · q 만 고릅니다. 요일 주기를 후보에 둘지는 셋째 날에 다룹니다.

호출은 이웃한 차수만 이동하며 비교하는 **단계별 탐색**으로 합니다. 후보를 빠짐없이 적합하는 전수 탐색은 지정하지 않습니다. 전수 탐색을 지정하면 AIC 가 달라져 기준 범위를 벗어납니다.

**출력 형식** — 값 2개를 「이름 = 값 단위」 형태로 **서로 다른 행에 한 행씩** `print` 합니다. 참고 행은 등호 없이 한 행 더 적습니다. 그래프는 한 장입니다.

**통과 기준** — 굵은 이름 2개가 각각 자기 행에 적혀 있고 그 값이 기준 범위에 들어야 합니다. 막대 21개와 경계선 2개와 가로축·세로축 이름까지 갖추면 통과입니다. 채점 화면은 이 경계선을 「가로선」이라 부릅니다.

<img src="https://otexts.com/fpp3/fpp_files/figure-html/beeracf-1.png" width="640" height="192"/>
<p align="center"><em>▲ 호주 분기별 맥주 생산량의 자기상관 — 한 주기가 4분기인 계열에서 시차 4 · 8 · 12 · 16 이 경계선 바깥으로 함께 커지고, 주기의 절반인 시차 2 · 6 · 10 · 14 는 음수 방향으로 경계선 바깥으로 벗어난다. 둘 다 같은 계절 구조다 (출처: Hyndman &amp; Athanasopoulos, Forecasting: Principles and Practice 3rd ed., otexts.com/fpp3)</em></p>

<img src="https://otexts.com/fpp3/fpp_files/figure-html/wnacfplus-1.png" width="640" height="192"/>
<p align="center"><em>▲ 자기상관이 없는 백색잡음 계열 3개의 자기상관 막대 — 왼쪽에서 오른쪽으로 자료 수가 늘면 경계선 사이가 좁아진다. 경계선은 1.96 을 자료 수의 제곱근으로 나눈 값이고, 막대가 그 안쪽에 있으면 자기상관이 0 이라고 본다 (출처: 같은 책)</em></p>

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름** — `daily_c` · `auto_arima` 는 이미 있으므로 다시 만들지 말라고 적습니다.
- **탐색 조건** — 계절 차수를 고르지 않는다는 것(`seasonal=False`), 기준 점수가 AIC 라는 것, 단계별 탐색을 쓴다는 것.
- **출력** — 값 2개의 이름과 단위, 소수점 아래 몇 자리까지 출력할지, 두 값을 서로 다른 행에 둔다는 것. 참고 행에 넣을 값과 등호를 쓰지 않는다는 것.
- **그래프** — 어느 계열의 몇 번째 시차부터 그릴지 · 강조할 요일 시차 · 경계선 2개 · 가로축과 세로축 이름.

**막히면** — AI 튜터에게 「자동 차수 탐색은 레벨 차분 횟수 d 를 무엇으로 정하나요」 처럼 질문하세요.

#### 예상 적기

하나를 고르고, 그렇게 판단한 이유를 한 문장 적으세요. 하루 합계 계열은 365일 동안 여름에 오르고 겨울에 내려가는 움직임이 큽니다.

> **[예측] 하루 합계 365일 계열에 자동 차수 탐색을 적용하면 레벨 차분 횟수 d 가 몇으로 나올까요?**
>
> 1. 0 — 차분하지 않아도 평균 수준이 일정하다
> 2. 1 — 하루 전 값을 한 번 빼야 평균 수준이 일정해진다
> 3. 2 — 하루 전 값을 두 번 빼야 평균 수준이 일정해진다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

**읽는 법** — 실행해 두 값과 그림을 받은 뒤에 아래 「자세히」 를 펼칩니다. d 가 1 이면 하루 합계 계열은 차분 없이는 평균 수준이 일정하지 않다는 뜻입니다. AIC 는 이 365일 계열에서만 뜻이 있는 점수입니다. 미션 2 의 시간당 수요 AIC 와 나란히 놓지 않습니다.

<details class="lms-extra"><summary>자세히 — 받은 그림과 요일 차이를 읽는 순서</summary>

그림에서 가장 먼저 보이는 막대는 시차 1 의 −0.50 입니다. 하루 합계에서 하루 전 값을 빼면, 어제 튄 값이 오늘은 반대 부호로 한 번 더 나타나 음수가 됩니다. 시스템이 멈춰 합계가 0 인 12일이 이 막대를 더 키웁니다. 휴무를 채운 보정 수요의 하루 합계로 같은 계산을 하면 시차 1 은 −0.32 입니다.

다음은 요일 시차입니다. 시차 7 이 −0.0352, 시차 14 가 +0.0380 으로 경계선 ±0.1027 안쪽입니다. 시차 21 만 +0.1059 로 겨우 경계선에 닿습니다.

요일 규칙이 뚜렷하다면 시차 7 · 14 · 21 이 함께 커져야 합니다. 요일 시차 셋 가운데 하나만 경계선 바깥인 것은 우연으로 보는 편이 자연스럽습니다. 시차 3 의 −0.18 과 시차 4 의 +0.13 도 경계선 밖이지만, 7 의 배수가 아니라 요일 규칙의 근거가 되지 않습니다.

요일 차이가 흐려진 것은 하루로 합친 결과입니다. **계절 강도 F_S** 는 계절 성분이 설명하는 비중을 0 에서 1 사이로 측정한 무차원 값이고, 1 에 가까울수록 그 주기가 강합니다. 첫날 이론이 같은 1년 구간에서 측정한 값은 주기 24 의 시간당 수요가 0.572, 주기 7 의 일 수요가 0.089 입니다.

보정 수요의 요일별 하루 합계 평균은 가장 큰 수요일 18,765대/일과 가장 작은 일요일 15,390대/일로 1.22배입니다. 같은 자료를 시각 단위로 보면 08시 평일 평균 1,320대와 주말 평균 424대로 3.11배입니다. 요일 차이는 하루로 합치는 순간 대부분 상쇄됩니다.

</details>

이 모형이 산출하는 예측 1개는 「그날 하루 합계」 하나입니다. 몇 시에 몇 대를 채울지는 여기서 나오지 않습니다. 그래서 다음 미션은 시간당 수요를 사용합니다.

<details class="lms-extra"><summary>배경 — 하루 합계 계열을 먼저 배제하는 이유와 자동 차수 탐색이 정하는 것</summary>

시간당 수요는 8,760시간이고 계절 주기가 24라 계산량이 큽니다. 그래서 「하루 합계로 집계한 365행으로 대신할 수 없는가」 를 먼저 확인해 배제합니다.

**자동 차수 탐색(auto_arima)** 은 이론의 「5. 후보 선택 기준 AIC 와 자동 차수 탐색」에서 다룬 도구입니다. 차분 횟수를 검정으로 먼저 결정하고, 남은 차수만 AIC 로 탐색해 고릅니다. 오늘은 `seasonal=False` 로 호출해 비계절 차수 p · d · q 만 고르게 합니다.

**경계선 ±1.96/√T 의 1.96 은 95% 수준에서 온 수입니다.** 하루 전 값을 뺀 이 계열은 365일에서 1개가 줄어 T = 364 이므로 경계선은 0.1 부근입니다. 막대가 그 안쪽이면 그 시차의 자기상관은 0 과 구별되지 않습니다.

**그래프를 하루 전 값을 뺀 계열로 그리는 이유도 같습니다.** 차분하지 않은 계열에는 추세가 남아 있어 시차를 얼마로 두든 자기상관이 크게 나옵니다. 하루 전 값을 뺀 뒤에야 요일 시차만 두드러지는지가 확인됩니다.

<img src="https://otexts.com/fpp3/fpp_files/figure-html/ARMAgridsearch-1.png" width="384" height="384"/>
<p align="center"><em>▲ 이 미션이 호출하는 자동 차수 탐색이 이동하며 비교하는 차수 격자 — 세로가 자기회귀 차수 p, 가로가 이동평균 차수 q 이고 색은 탐색 단계, 검은 동그라미는 그 단계의 최선이다. 단계별 탐색은 이렇게 이웃 조합만 이동한다 (출처: Hyndman &amp; Athanasopoulos, Forecasting: Principles and Practice 3rd ed., 그림 9.11)</em></p>

</details>

<details class="lms-extra"><summary>자세히 — AI 튜터에게 맡길 일과 내가 확인할 일</summary>

AI 튜터가 코드를 입력하는 위치는 마지막으로 클릭한 코드 셀이라, 다른 셀을 클릭한 채 지시하면 엉뚱한 셀이 바뀝니다. 아래 왼쪽은 맡겨도 되는 일이고, 오른쪽은 **틀려도 오류가 발생하지 않아** 사람이 확인해야 하는 일입니다.

| AI 튜터에게 맡길 일 | 내가 반드시 확인할 일 |
|---|---|
| 하루 합계로 집계하는 코드 | 집계 대상이 원본 시간 계열인가 |
| 자동 차수 탐색 호출과 인자 채우기 | 계절 차수를 고르지 않는 설정으로 호출했는가 |
| 막대 21개를 그리고 경계선 긋기 | 경계선이 위아래 2개인가, 막대가 시차 1 부터 시작하는가 |

</details>

In [ ]:
# 미션 1 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 1 에 쓸 변수를 직접 만듭니다
URL = "https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip"
raw = pd.read_csv(URL, encoding="cp1252", compression="zip")        # 8,760행 x 14열
#  받아지지 않으면: UCI 페이지에서 zip 을 내려받아 Colab 좌측 파일 패널에 올린 뒤
#  URL 대신 "seoul+bike+sharing+demand.zip" 을 넣으세요

raw.columns = ["date", "count", "hour", "temp", "humidity", "wind", "visibility",      # 위치 기반 리네임
               "dewpoint", "solar", "rain", "snow", "season", "holiday", "functioning"]
raw["datetime"] = (pd.to_datetime(raw["date"], format="%d/%m/%Y")    # 01/12/2017 = 2017년 12월 1일
                   + pd.to_timedelta(raw["hour"], unit="h"))
df = raw.sort_values("datetime").set_index("datetime")
df.index.freq = "h"                       # 등간격 1시간 — 시차 24 가 "24시간 전" 이라는 뜻이 된다

_s = df["count"].astype(float).mask(df["functioning"].eq("No"))   # 휴무는 0 이 아니라 관측 없음
_y = _s.fillna(_s.shift(24 * 7))          # 1주일 전 같은 시각 값으로 채운다 (요일 효과를 지키려고)
_y = _y.fillna(_y.shift(-24 * 7))         # 1주일 전도 휴무인 시각만 1주일 뒤 값으로
df["demand"] = _y                         # 보정 수요 — 모형 학습이 쓸 열 (원본 시간 계열 count 는 그대로)
df["is_imputed"] = df["functioning"].eq("No")

daily_c = df["count"].resample("D").sum()   # 쓸 변수 — 원본 시간 계열의 하루 합계로 집계한 일 수요

try:                                        # 자동 차수 탐색 — 없으면 같은 이름의 대체를 준비한다
    from pmdarima import auto_arima
    _search_backend = "pmdarima.auto_arima"
except Exception:
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pmdarima==2.1.1"], check=False)
    try:
        from pmdarima import auto_arima
        _search_backend = "pmdarima.auto_arima (이 셀에서 설치)"
    except Exception:
        auto_arima = auto_arima_like          # 대체 탐색 — 정의는 맨 위 [채점] 셀에 있습니다
        _search_backend = "auto_arima_like (statsmodels 로 만든 대체)"

print(f"행 x 열 : {df.shape[0]:,} x {df.shape[1]}   기간 {df.index.min()} ~ {df.index.max()}")
print(f"원본 시간 계열 count 평균 {df['count'].mean():.4f} 대/시간  →  "
      f"보정 수요 demand 평균 {df['demand'].mean():.4f} 대/시간")
print(f"휴무 {int(df['is_imputed'].sum())}시간 / "
      f"{df.index[df['is_imputed']].normalize().nunique()}일   "
      f"보정 수요 결측 {int(df['demand'].isna().sum())}건")
print(f"일 수요 {len(daily_c)}일   평균 {daily_c.mean():,.1f} 대/일")
print(f"그중 시스템이 정지해 하루 합계가 0 인 날 {int((daily_c == 0).sum())}일   "
      f"탐색과 검정은 원본 시간 계열로 한다는 첫날의 약속을 따른 결과입니다")
print(f"자동 차수 탐색 : {_search_backend}")
if "auto_arima_like" in _search_backend:
    print("ⓘ 대체 탐색은 pmdarima 와 다른 차수를 골라 AIC 가 달라집니다 — 채점은 AIC 칸을 보지 않습니다")
print("쓸 수 있는 이름 : df (표 전체) · df['count'] (원본 시간 계열, 대)"
      " · daily_c (하루 합계 일 수요, 대/일)"
      " · auto_arima (자동 차수 탐색 함수)")

In [ ]:
# 미션 1 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

> **[문제] 미션 1 은 자동 차수 탐색을 `seasonal=False` 로 호출합니다. 그 설정이 하는 일로 맞는 것은?**
>
> 1. 하루 합계 계열에서 계절 성분을 먼저 빼고 남은 계열만 탐색에 넘긴다 — 계열 자체가 달라진다
> 2. 계절 주기를 선언하지 않아 탐색이 비계절 차수 p · d · q 만 고르게 한다 — 계열은 365일 그대로다
> 3. 계절 차수를 모두 0 으로 고정한 뒤 그 값을 결과에 함께 적는다 — 그래서 주기 값도 함께 출력된다

> **[문제] 미션 1 의 「참고」 행은 계절 차수를 (0, 0, 0, 0) 으로 출력합니다. 뒤 괄호 (P, D, Q, m) 의 가운데 값 D 와 마지막 값 m 이 뜻하는 것으로 맞는 것은?**
>
> 1. D 는 계절차분 횟수, m 은 계절 주기다 — 둘 다 0 이라 계절차분도 주기 선언도 없다
> 2. D 는 계절 차수의 개수, m 은 탐색이 적합한 후보의 개수다 — 둘 다 0 이라 적합이 없었다
> 3. D 는 레벨 차분 횟수, m 은 계열의 길이다 — 앞 괄호의 가운데 값과 같은 뜻이다

## 2. 남은 차수 2개의 결정 — AIC 비교표

### 미션 2 [필수] — 남은 차수 2개를 AIC 로 결정하기 (함정 미션)

#### 할 일 — 미션 2 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 「PROMPT 셀」을 먼저 그대로 ▶ 해 전임 담당자 코드의 채점 결과(❌)를 봅니다. 고치는 것은 3번부터입니다. PROMPT 가 비어 있어도 이 채점은 출력됩니다.
3. 아래 고칠 것대로 PROMPT 를 쓰고, 미션 1 의 3번과 4번과 같은 순서로 코드를 받아 실행합니다.

**이 미션의 질문** — 후보 6조합의 AIC 를 한 표에 배치해 비교하려면 어디를 고쳐야 합니까?

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `y_train` | 학습 구간 보정 수요 1,848시간, 단위 대/시간 | 고정 |
| `D_LEVEL` · `D_SEASONAL` · `M` | 차분 조건 값 3개 — 레벨 차분 0 · 계절차분 1 · 계절 주기 24 | 고정 |
| `P_GRID` | 비계절 자기회귀 차수 p 후보 3개 | 고정 |
| `Q_GRID` | 비계절 이동평균 차수 q 후보 2개 | 고정 |
| `FIT_OPTS` | 적합 옵션 문자열 | 그대로 적는다 |

후보 격자 p 0～2 · q 0～1 은 이론에서 좁힌 후보 넷을 모두 담는 범위입니다. 이론이 규칙대로 읽은 p = 3 은 격자에 넣지 않았습니다. 이 1,848시간에서 (3,0,0) · (3,0,1) 의 AIC 는 격자 안 1위보다 3 넘게 높습니다.

#### 고칠 것

전임 담당자의 코드는 오류 없이 실행되지만 값이 틀렸습니다. 고친 뒤 아래가 모두 출력되게 합니다. 굵은 이름은 글자 그대로 사용합니다.

1. **공식 차수 AIC** — 단위 없는 점수 · 소수점 아래 2자리 · AIC 가 가장 낮은 조합의 AIC
   - 고칠 곳 : 전임 코드가 차분 조건으로 적어 둔 값을 「준비 셀」이 출력한 고정값 3개와 하나씩 대조합니다. 어긋난 값이 코드 여러 곳에 적혀 있으면 모두 고칩니다
   - 차수를 출력하는 「공식 차수 = …」 행도 고친 차수로 적습니다
2. **1위와 2위의 AIC 차이** — 단위 없는 점수 · 소수점 아래 2자리 · AIC 가 가장 낮은 조합과 그다음 조합의 AIC 차이
   - 두 행의 차분 조건이 같아야 이 차이에 뜻이 있습니다
3. **그래프 한 장** — 가로축은 AIC, 세로축은 차수 라벨 6개인 가로 막대 그래프
   - 막대 : 후보 6조합에 하나씩, 모두 6개
   - 가로축 이름 : 「AIC (낮을수록 좋음)」 · 세로축 이름 : 「차수 (비계절)(계절)」
   - 세로 눈금 : 조합마다 차수 라벨 하나 — 「공식 차수」 행과 같은 모양으로, 고친 값으로 적습니다
   - 채점 대상은 아니지만 이렇게 두면 읽기 좋습니다 : AIC 가 낮은 것이 위로 오게 정렬 · 가장 낮은 막대 하나만 다른 색
   - 가로축 범위 25,000 부터 28,000 까지

**출력 형식** — 값 2개를 「이름 = 값」 형태로 서로 다른 행에 한 행씩 `print` 합니다. 그래프는 한 장입니다.

**통과 기준** — 굵은 이름 2개가 각각 자기 행에 적혀 있고 그 값이 기준 범위에 들며, 가로 막대 6개와 가로축·세로축 이름을 갖추면 통과입니다. 「공식 차수」 행과 세로 눈금의 차수 표기도 고친 차수여야 합니다. 적합만 고치고 출력 문자열을 그대로 두면 AIC 가 맞아도 통과하지 못합니다.

**읽는 법** — 고친 표를 AIC 순으로 정렬해 1위와 2위를 봅니다. 이론이 정한 기준대로 두 AIC 의 차이가 2 이하이면 두 후보를 구별할 수 없다고 보고, 항이 적은 쪽을 공식 차수로 고릅니다. 그렇게 고른 차수를 내일 「준비 셀」이 그대로 인용합니다.

이론의 336시간과는 자료 길이가 달라 AIC 값 자체가 다르므로, 비교는 이 표 안에서만 합니다.

다만 AIC 1위는 후보일 뿐이고, 운영에 쓸지는 내일 검증 168시간의 RMSE 를 기준선과 비교해 결정합니다.

#### 프롬프트에 넣을 것(힌트)

- **고칠 곳** — 대조에서 어긋난 값이 어느 코드 행에 있고 무엇으로 바꿀지 한 구로 적습니다. 같은 값이 차수를 출력하는 `print` 와 막대 라벨 문자열에도 적혀 있으면 그곳도 함께 바꾸라고 적습니다.
- **손대지 않을 범위** — `y_train` 과 후보 2개(`P_GRID` · `Q_GRID`)는 다시 만들지 말고, 이미 그려진 가로 막대도 같은 형태로 유지하라고 적습니다.
- **출력** — 값 2개의 이름, 그리고 소수점 아래 몇 자리까지 출력할지.
- **그래프** — 막대 6개 · 가로축과 세로축 이름.

**막히면** — AI 튜터에게 「차분 횟수가 다른 모형끼리 AIC 를 비교할 수 없는 이유가 무엇인가요」 처럼 질문하세요.

#### 틀린 곳 찾기

전임 담당자 코드를 그대로 실행하면 표 6행과 값 3행이 출력됩니다. 그중 **위 2행(공식 차수 · 공식 차수 AIC)** 을 아래 빈칸에 옮겨 적고, 틀렸다고 판단한 곳에 근거 한 문장을 함께 적으세요. 고친 뒤 그 2행이 어떻게 달라지는지가 답입니다.

> **전임 코드의 차수와 AIC:** ____

> **[예측] 「여기부터 AI 코드」 아래 코드는 실행되는데 값이 틀렸습니다. 틀린 곳은 어디일까요?**
>
> 1. `order` 튜플의 가운데 값이 1 이다 — 6조합 전부에 레벨 차분을 한 번씩 적용하고 있다
> 2. `seasonal_order` 의 마지막 값이 24 다 — 시간당 수요에 다른 주기를 전달했다
> 3. 학습 구간이 「준비 셀」의 계열 하나다 — 조합마다 다른 구간을 써야 한다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

<details class="lms-extra"><summary>배경 — 첫날이 정한 차수 3개와 오늘 정할 차수 2개</summary>

<img src="https://otexts.com/fpp3/fpp_files/figure-html/traintest-1.png" width="640" height="64"/>
<p align="center"><em>▲ 계열을 학습 구간과 검증 구간으로 나누는 방식 — 파란 점이 모형이 계수를 추정하는 학습 구간이고, 주황 점이 오차를 측정하는 검증 구간이다. 오늘의 AIC 는 왼쪽 1,848시간에서만 계산하고, 오른쪽에 해당하는 168시간은 내일 사용한다 (출처: Hyndman &amp; Athanasopoulos, Forecasting: Principles and Practice 3rd ed., 5.8)</em></p>

SARIMA 가 사용하는 숫자는 비계절 p · d · q 3개와 계절 P · D · Q · m 4개, 모두 7개입니다. 그중 3개는 첫날의 검정과 계절 강도에서 이미 결정되었습니다.

| 숫자 | 값 | 첫날의 근거 |
|---|---|---|
| 계절 주기 m | 24 | 첫날 1년 구간에서 측정한 계절 강도는 시각 주기 24 가 0.572, 요일 주기 7 이 0.089 였다. 그래서 하루 주기 24 를 계절 주기로 골랐다 |
| 계절차분 횟수 D | 1 | 원본은 ADF −6.95 · KPSS 8.005 로 둘 다 기각이라 두 답의 방향이 반대였다. 그래서 남은 구조를 ACF 로 보고, 주기 24 가 남아 있어 24시간 계절차분을 골랐다 |
| 레벨 차분 횟수 d | 0 | 24시간 계절차분 뒤 다시 검정하니 ADF −17.0 · KPSS 0.008 로 둘 다 정상 쪽이었다. 그래서 레벨 차분은 더 하지 않았다 |

남은 4개 가운데 계절 P 와 Q 는 셋째 날에 정합니다. **오늘 AIC 로 비교하는 것은 비계절 p 와 q 2개입니다.** AIC 는 이론의 「5. 후보 선택 기준 AIC 와 자동 차수 탐색」에서 정의한 점수이고, 낮을수록 좋으며 **차분 조건이 다른 모형끼리는 비교할 수 없습니다.**

차분을 한 번 더 적용하면 그 계열이 관측될 그럴듯함, 곧 **우도**를 계산한 계열 자체가 달라지기 때문입니다. 아래 두 그림이 그 「달라진다」입니다.

<img src="https://otexts.com/fpp3/fpp_files/figure-html/usemployment2-1.png" width="640" height="352"/>
<p align="center"><em>▲ 미국 고용 계열에 12개월 계절차분만 한 번 적용한 계열(위)과 그 자기상관·부분자기상관(아래) — 첫날 실습에서 본 그림이다 (출처: Hyndman &amp; Athanasopoulos, Forecasting: Principles and Practice 3rd ed., 9.1)</em></p>

<img src="https://otexts.com/fpp3/fpp_files/figure-html/usemployment3-1.png" width="640" height="352"/>
<p align="center"><em>▲ 같은 계열에 계절차분과 레벨 차분을 잇달아 적용한 계열 — 위 그림과 세로축 눈금도, 아래 자기상관 막대의 모양도 다르다. 두 그림은 서로 다른 계열이고, 각각에서 계산한 AIC 를 한 표에서 비교할 수 없다 (출처: 같은 책, 9.1)</em></p>

</details>

<img src="https://otexts.com/fpp3/fpp_files/figure-html/h02b-1.png" width="640" height="395"/>
<p align="center"><em>▲ 로그를 취한 뒤 계절차분한 계열(위)과 그 자기상관·부분자기상관(아래) — 왼쪽 자기상관은 시차 9 까지 천천히 줄고, 오른쪽 부분자기상관은 시차 1～3 이 경계선 밖이고 그 뒤 비계절 시차에서는 안쪽으로 들어간다. 그래서 비계절 자기회귀 차수 p 의 후보는 3 이다. 계절차분을 적용한 뒤에도 이런 단서가 남는다 (출처: 같은 책, 9.9)</em></p>

In [ ]:
# 미션 2 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 2 에 쓸 변수를 직접 만듭니다
if "df" not in globals() or "demand" not in df.columns:   # 이 미션만 따로 열어도 단독으로 실행됩니다
    URL = "https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip"
    raw = pd.read_csv(URL, encoding="cp1252", compression="zip")      # 8,760행 x 14열
    #  받아지지 않으면: UCI 페이지에서 zip 을 내려받아 Colab 좌측 파일 패널에 올린 뒤
    #  URL 대신 "seoul+bike+sharing+demand.zip" 을 넣으세요
    raw.columns = ["date", "count", "hour", "temp", "humidity", "wind", "visibility",   # 위치 기반 리네임
                   "dewpoint", "solar", "rain", "snow", "season", "holiday", "functioning"]
    raw["datetime"] = (pd.to_datetime(raw["date"], format="%d/%m/%Y")   # 01/12/2017 = 2017년 12월 1일
                       + pd.to_timedelta(raw["hour"], unit="h"))
    df = raw.sort_values("datetime").set_index("datetime")
    df.index.freq = "h"                   # 등간격 1시간 — 시차 24 가 "24시간 전" 이라는 뜻이 된다
    _s = df["count"].astype(float).mask(df["functioning"].eq("No"))   # 휴무는 0 이 아니라 관측 없음
    _y = _s.fillna(_s.shift(24 * 7))      # 1주일 전 같은 시각 값으로 채운다 (요일 효과를 지키려고)
    df["demand"] = _y.fillna(_y.shift(-24 * 7))   # 1주일 전도 휴무인 시각만 1주일 뒤 값으로
    df["is_imputed"] = df["functioning"].eq("No")

FIT_START, TRAIN_END = "2018-09-01 00:00", "2018-11-16 23:00"
y_train = df["demand"].loc[FIT_START:TRAIN_END]   # 쓸 변수 — 학습 구간의 보정 수요 (대/시간)

D_LEVEL, D_SEASONAL, M = 0, 1, 24         # 차분 조건 — 첫날의 검정과 계절 강도가 정한 값 3개
P_GRID, Q_GRID = [0, 1, 2], [0, 1]        # 비계절 자기회귀·이동평균 후보
FIT_OPTS = "trend=None, enforce_stationarity=True, enforce_invertibility=True"

print(f"학습 구간 {FIT_START} ~ {TRAIN_END}   {len(y_train):,}시간   "
      f"채운 시각 {int(df['is_imputed'].loc[FIT_START:TRAIN_END].sum())}시간")
print(f"학습 평균 {y_train.mean():,.1f} 대/시간   결측 {int(y_train.isna().sum())}건")
print(f"차분 조건 고정 — 레벨 차분 d = {D_LEVEL} · 계절차분 D = {D_SEASONAL} · 계절 주기 m = {M}")
print(f"계절 차수 고정 — (0, {D_SEASONAL}, 0, {M})   오늘 비교하는 것은 p 와 q 뿐입니다")
print(f"후보 격자 — p {P_GRID} x q {Q_GRID}   조합 {len(P_GRID) * len(Q_GRID)}개")
print(f"적합 옵션 통일 — {FIT_OPTS}")
print("여섯 조합을 모두 적합하는 데 1분이 채 걸리지 않습니다.")
print("쓸 수 있는 이름 : y_train (학습 구간 보정 수요, 대/시간) · D_LEVEL · D_SEASONAL · M"
      " (차분 조건 값 3개) · P_GRID (비계절 p 후보) · Q_GRID (비계절 q 후보)"
      " · FIT_OPTS (적합 옵션 문자열)")

In [ ]:
# 미션 2 — 프롬프트 (함정)   # ai: prompt
# 이 미션은 함정입니다 — 아래는 전임 담당자가 남긴 코드라 돌아가지만 답이 틀렸습니다.
# 어디를 어떻게 고칠지 PROMPT 에 쓰면 튜터가 AI 코드 표시선 아래를 다시 씁니다.
PROMPT = """

"""
# --- 여기부터 AI 코드 ---
# ↓ 전임 담당자의 코드 — 그대로 실행하면 채점이 ❌ 를 냅니다
rows = []
for p, q in itertools.product(P_GRID, Q_GRID):
    _r = SARIMAX(y_train, order=(p, 1, q), seasonal_order=(0, D_SEASONAL, 0, M),
                 trend=None, enforce_stationarity=True,
                 enforce_invertibility=True).fit(disp=False)
    rows.append({"p": p, "q": q, "AIC": _r.aic})

grid = pd.DataFrame(rows).sort_values("AIC").reset_index(drop=True)
best = grid.iloc[0]
second = grid.iloc[1]
print(grid.round(2).to_string(index=False))
print(f"공식 차수 = ({int(best.p)},1,{int(best.q)})(0,{D_SEASONAL},0,{M})")
print(f"공식 차수 AIC = {best.AIC:,.2f}")
print(f"1위와 2위의 AIC 차이 = {second.AIC - best.AIC:,.2f}")

labels = [f"({int(r.p)},1,{int(r.q)})(0,{D_SEASONAL},0,{M})" for r in grid.itertuples()]
colors = [COL["하늘색"] if i == 0 else COL["연회색"] for i in range(len(grid))]
fig, ax = plt.subplots(figsize=(FIG_W, 4.4))
ax.barh(range(len(grid) - 1, -1, -1), grid["AIC"], color=colors)
ax.set_yticks(range(len(grid) - 1, -1, -1))
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlim(25000, 28000)
ax.set_xlabel("AIC (낮을수록 좋음)")
ax.set_ylabel("차수 (비계절)(계절)")
plt.show()

> **[문제] 미션 2 에서 고친 뒤의 코드는 `order` 튜플의 가운데 값을 6조합 모두 같은 값으로 둡니다. 그 값이 조합마다 달라지면 AIC 표에 무슨 일이 생기나요?**
>
> 1. 값의 크기만 조금 달라진다 — 순위는 그대로라 1위 차수는 바뀌지 않는다
> 2. 행마다 우도를 계산한 계열이 달라져 AIC 열이 서로 비교할 수 없는 점수가 된다 — 1위 차수까지 바뀔 수 있다
> 3. 차분을 더 한 조합은 관측 하나를 잃어 패널티 항이 커지므로 AIC 가 반드시 나빠진다

> **[문제] 미션 2 의 고친 표에서 1위는 (1,0,1)(0,1,0,24), 2위는 (2,0,1)(0,1,0,24) 입니다. 출력한 「1위와 2위의 AIC 차이」 는 1.83 입니다. 이 값을 읽는 방법으로 맞는 것은?**
>
> 1. 1위가 2위보다 낫다는 근거다 — AIC 는 낮을수록 좋으므로 차이의 크기와 무관하게 순위를 그대로 따른다
> 2. 두 모형의 예측이 시각마다 평균 1.83대 만큼 어긋난다는 뜻이다
> 3. 두 차수를 구별할 수 없다는 뜻이라 자기회귀 차수가 하나 적은 (1,0,1) 을 고른다

## 3. 오늘의 정리와 내일 예고 — 정한 차수로 168시간 채우기

## 오늘의 정리

배차팀장이 요청한 168시간을 채울 모형의 차수가 정해졌습니다. 아래 표는 오늘의 결과를 질문별로 나눈 것입니다.

| 오늘의 질문 | 우리가 얻은 답 |
|---|---|
| 하루 합계 계열로 대신할 수 있나 | 대신할 수 없습니다. 하루로 합치면 요일 차이가 1.22배로 흐려지고, 예측 1개가 하루 합계 하나라 시간대별 배차의 근거도 되지 않습니다 |
| 168시간을 채울 차수는 무엇인가 | 첫날이 정한 차분 조건을 6조합에 똑같이 적용한 뒤 AIC 로 골랐습니다. 공식 차수는 (1,0,1)(0,1,0,24) 이고 그 AIC 는 25,209.34 입니다 |
| 1위와 2위를 어떻게 갈랐나 | 차이가 1.83 이라 두 후보를 구별할 수 없어, 비계절 자기회귀 차수가 하나 적은 (1,0,1) 을 골랐습니다 |

한 문장으로 답하면 이렇습니다. **다음 주 168시간은 차수 (1,0,1)(0,1,0,24) 의 모형으로 채웁니다.**

AIC 표에서 1위와 2위의 차이는 1.83 이었습니다. 차분 조건을 한 곳만 어긋나게 적어도 순서가 반대가 되는 크기입니다. 그래서 차수와 AIC 를 문서에 남길 때 「레벨 차분 0 · 계절차분 1 · 계절 주기 24 로 6조합 통일」이라는 조건을 함께 적습니다.

## 내일 예고 — 이 차수로 168시간을 채우고 기준선과 비교합니다

AIC 가 가장 낮은 차수는 후보일 뿐입니다. AIC 는 학습 구간 1,848시간에서 계산한 점수라, 아직 보지 않은 168시간을 얼마나 맞히는지는 말해 주지 않습니다. 내일은 오늘 고른 (1,0,1) 에 계절 항 (1,1,1,24) 를 더합니다. AIC 가 얼마나 내려가는지는 그때 봅니다. 그 차수로 학습 구간 끝에서 168시간을 한 번에 예측하고, 1주일 전 같은 시각 값을 그대로 산출하는 기준선과 오차를 비교합니다. 예측값 옆에는 95% 예측구간을 적어, 시각마다 그 숫자를 배차에 그대로 쓸 수 있는지까지 표에 남깁니다.

<img src="https://otexts.com/fpp3/fpp_files/figure-html/usemployment7-1.png" width="640" height="320"/>
<p align="center"><em>▲ 계절 패턴을 이어 그리는 SARIMA 예측 — 파란 선이 점 예측이고, 점 예측을 가운데 두고 진한 영역이 80% · 옅은 영역이 95% 예측구간이다. 지평이 멀어질수록 예측구간이 길어진다. 내일 만들 표가 이 그림을 숫자로 적은 것이다 (출처: Hyndman &amp; Athanasopoulos, Forecasting: Principles and Practice 3rd ed., 9.9)</em></p>

내일 「준비 셀」이 오늘의 차수 (1,0,1)(0,1,0,24) 를 그대로 사용합니다. 미션 2 의 출력에서 차수와 AIC 두 행을 적어 두세요.

마무리로 완성본과 내 코드가 달랐던 부분 1곳과 거기서 배운 것 한 문장을 적어 팀 채널에 공유합니다.

<div style="background:#F2F2F2;color:#000000;padding:28px;border-radius:12px;border:1px solid #E6E6E6;margin-top:16px;border-left:4px solid #051C2C">
<h2 style="color:#000000;margin:0 0 8px 0">수고했습니다 — 차수 결정 완료</h2>
<div style="color:#656565">하루 합계 계열을 배제하고, 후보 6조합의 AIC 를 한 표에서 비교해 차수를 골랐습니다.<br/>내일은 이 차수를 출발점으로 168시간을 채웁니다.</div>
</div>